# Tomahawk Rev2.5 — CAD-A2-resolved Kaggle screening

Use exactly these two files: `Tomahawk_CAD_A2_Resolved_Kaggle.ipynb` and `Tomahawk_CAD_A2_Resolved_Data_v2_0.zip`. Attach only the ZIP and choose **Run All**. The complete runner is embedded below; no `.py`, package, internet access, or older Tomahawk input is required.

This release uses the 100-turn reference count recovered from the Kaggle output, but it does not reuse the legacy current on CAD-A2. It runs a separate model-derived current rerun on the exact CAD-A2 centerline and makes that ripple result a controlling screening gate. The legacy branch is retained only as an advisory comparison.

The coolant screen embeds verified IAPWS-IF97 Region 1, IAPWS 2008 viscosity, and IAPWS 2011 conductivity branches with official checkpoint tests; constant-property fallback is disabled. PF/shaping coil data and free-boundary mgrid are still reported as explicit blockers because they are not present in the supplied CAD/Kaggle inputs.

The notebook is designed to complete without throwing when a physics gate fails. Expected runtime markers are `KAGGLE_BOOTSTRAP_START`, `ONE_CLICK_INPUT_VERIFIED`, `SMOKE_COMPLETE`, `FULL_SCREEN_COMPLETE`, and `RUN_COMPLETE`. The physical result is read from `gate_results.json`; this is a screening artifact, not engineering validation.

In [ ]:
#!/usr/bin/env python3
"""Tomahawk Rev2.5 CAD-A2 closure screening.

This program is a reproducible screening layer for the current Rev1.7
provisional input bundle.  It performs:

* a fixed-boundary, axisymmetric Grad-Shafranov solve;
* D-T reaction-rate, stored-energy, bremsstrahlung, alpha-heating, and
  confinement-loss calculations using the Bosch-Hale reactivity fit;
* consistency and conservation checks on the supplied profile;
* a first-pass RF-cassette coolant hydraulic/energy screen using verified
  IAPWS-IF97 Region-1, IAPWS-2008 viscosity, and IAPWS-2011 conductivity
  correlations; and
* a CAD-A2-consistent toroidal-field ripple screen using a remapped,
  model-derived 100-turn current schedule from the Kaggle output;
* a reduced uncertainty sweep.

It deliberately does not claim a free-boundary 3-D equilibrium, nonlinear MHD
stability, orbit-resolved alpha transport, OpenMC/DAGMC tallies, CFD/CHT, or a
qualified CHF result.  Those items are emitted as explicit external-solver
blockers.  All numerical pass/fail labels in this file are screening labels.

The input bundle is expected to contain:
    MHD/mhd_input.json
    MHD/mhd_profile_sweep.csv
    CFD/cfd_cht_parameter_sweep.csv
    CHF/chf_input.json
    MHD/cad_a2_winding_envelope.csv
    MHD/cad_a2_coil_centerlines.csv
    MHD/cad_a2_coil_currents.csv
    MHD/cad_kaggle_reconciliation.json
    MHD/water_property_sources.json

The runner is fail-closed: it verifies the dataset manifest and SHA-256 map,
requires the corrected Rev2.4 confinement/coolant fields, and never substitutes
a fallback profile for a missing or malformed input.

NumPy is required.  SciPy is used for the fast sparse Grad-Shafranov solve
when available; a NumPy SOR fallback is retained for portability.  Matplotlib
is optional and only affects plot generation.
"""
from __future__ import annotations

import argparse
import csv
import hashlib
import json
import math
import os
from pathlib import Path
import platform
import sys
import time
import zipfile

import numpy as np


VERSION = "TOMAHAWK-REV2.5-CAD-A2-CLOSURE-1.0-KAGGLE"
MODEL_VERSION = "CAD-A2-WHOLE-CONCEPT-1.7-MULTIPHYSICS-DOMAINS"
EXPECTED_ACTIVE_TAU_E_S = 2.70
EXPECTED_REFERENCE_TAU_E_S = 1.36
EXPECTED_REFERENCE_SELF_HEATING_RATIO = 0.63
EXPECTED_FULL_INCIDENT_SECONDARY_FRACTION = 0.50
REQUIRED_INPUT_FILES = (
    "MHD/mhd_input.json",
    "MHD/mhd_profile_sweep.csv",
    "CFD/cfd_cht_parameter_sweep.csv",
    "CHF/chf_input.json",
    "MHD/cad_a2_winding_envelope.csv",
    "MHD/cad_a2_coil_centerlines.csv",
    "MHD/cad_a2_coil_currents.csv",
    "MHD/cad_kaggle_reconciliation.json",
    "MHD/water_property_sources.json",
    "dataset_manifest.json",
)
MU0 = 4.0e-7 * math.pi
KEV_TO_J = 1.0e3 * 1.602176634e-19
MEV_TO_J = 1.0e6 * 1.602176634e-19
E_FUSION_J = 17.6 * MEV_TO_J
ALPHA_FRACTION = 3.5 / 17.6
NEUTRON_FRACTION = 14.1 / 17.6

# Bosch-Hale D-T Maxwellian reactivity coefficients.  C1..C7 are the
# published D-T fit coefficients; the fit returns cm^3/s and is converted to
# m^3/s below.
BH_DT = {
    "B_G_sqrt_keV": 34.3827,
    "mrc2_keV": 1124656.0,
    "C1": 1.17302e-9,
    "C2": 1.51361e-2,
    "C3": 7.51886e-2,
    "C4": 4.60643e-3,
    "C5": 1.35000e-2,
    "C6": -1.06750e-4,
    "C7": 1.36600e-5,
    "source": "Bosch and Hale, Nuclear Fusion 32 (1992) 611-631",
}

DEFAULTS = {
    "R0_m": 7.5,
    "a_m": 3.0,
    "kappa": 1.0,
    "CAD_volume_m3": 950.0,
    # This is an explicit plasma-axis screening assumption.  It is not
    # inferred from the Rev1.7 35 T target, which is retained as Bpeak here.
    "B0_plasma_T": 14.0,
    "Bpeak_coil_T": 35.0,
    "Z_eff": 1.0,
    "tau_E_s": 2.70,
    "reference_condition_tau_E_s": 1.36,
    "alpha_deposition_fraction": 0.90,
    "auxiliary_absorbed_MW": 20.0,
    "rf_face_area_m2": 2.5,
    "water_roughness_m": 1.0e-5,
    "heated_length_m": 2.318,
    "pressure_drop_limit_Pa": 1.0e5,
    "CuCrZr_temperature_limit_K": 573.15,
    "minimum_DNBR": 1.30,
    # Literature range for acceptable TF ripple at the plasma edge is
    # roughly 0.3-0.5% (ITER; de Vries et al., Nucl. Fusion 48, 035007,
    # 2008) up to ~2% in tight-aspect-ratio compact devices before
    # mitigation is required (FAST, ORNL-EPR). This screening threshold
    # is a provisional, adjustable value, not a certified design limit.
    "tf_ripple_screening_max_percent": 1.0,
    "GS_residual_tolerance": 2.0e-3,
    "divB_relative_tolerance": 2.0e-10,
    "q95_proxy_min": 2.0,
    "beta_N_proxy_max": 3.5,
    "beta_axis_proxy_max_percent": 5.0,
    "tau_E_margin_min": 0.20,
    # A stored reference result may be compared against the current profile,
    # but it is never substituted for the current recomputation.
    "reference_self_heating_reconciliation_tolerance": 0.02,
    "secondary_heat_removal_fraction_by_scenario": {},
    "secondary_heat_removal_provenance": (
        "No secondary heat split supplied; cassette-local heat is used."),
}


# ---------------------------------------------------------------------------
# Verified ordinary-water property correlations used by the coolant screen.
#
# The coefficients below are transcribed from the official IAPWS releases
# listed in WATER_PROPERTY_SOURCES.  The operating states in this screen are
# subcooled liquid in Region 1; the implementation therefore intentionally
# rejects states outside the declared Region-1 validity range instead of
# extrapolating or silently falling back to constant properties.
# ---------------------------------------------------------------------------
WATER_PROPERTY_SOURCES = {
    "if97_region_1": (
        "https://iapws.org/technical-guidance/release/IF97-Rev.download"),
    "viscosity_2008": (
        "https://iapws.org/technical-guidance/release/viscosity.download"),
    "thermal_conductivity_2011": (
        "https://iapws.org/technical-guidance/release/ThCond.download"),
}
IF97_R1_R_J_KG_K = 461.526
IF97_R1_TSTAR_K = 1386.0
IF97_R1_PSTAR_PA = 16.53e6
IF97_R1_I = np.array(
    [0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 1, 1, 1, 1, 2, 2, 2, 2,
     2, 3, 3, 3, 4, 4, 4, 5, 8, 8, 21, 23, 29, 30, 31, 32],
    dtype=int)
IF97_R1_J = np.array(
    [-2, -1, 0, 1, 2, 3, 4, 5, -9, -7, -1, 0, 1, 3, -3, 0,
     1, 3, 17, -4, 0, 6, -5, -2, 10, -8, -11, -6, -29, -31,
     -38, -39, -40, -41], dtype=int)
IF97_R1_N = np.array([
    0.14632971213167,
    -0.84548187169114,
    -0.37563603672040e1,
    0.33855169168385e1,
    -0.95791963387872,
    0.15772038513228,
    -0.16616417199501e-1,
    0.81214629983568e-3,
    0.28319080123804e-3,
    -0.60706301565874e-3,
    -0.18990068218419e-1,
    -0.32529748770505e-1,
    -0.21841717175414e-1,
    -0.52838357969930e-4,
    -0.47184321073267e-3,
    -0.30001780793026e-3,
    0.47661393906987e-4,
    -0.44141845330846e-5,
    -0.72694996297594e-15,
    -0.31679644845054e-4,
    -0.28270797985312e-5,
    -0.85205128120103e-9,
    -0.22425281908000e-5,
    -0.65171222895601e-6,
    -0.14341729937924e-12,
    -0.40516996860117e-6,
    -0.12734301741641e-8,
    -0.17424871230634e-9,
    -0.68762131295531e-18,
    0.14478307828521e-19,
    0.26335781662795e-22,
    -0.11947622640071e-22,
    0.18228094581404e-23,
    -0.93537087292458e-25,
], dtype=float)

IAPWS08_VISCOSITY_H0 = np.array(
    [1.67752, 2.20462, 0.6366564, -0.241605], dtype=float)
IAPWS08_VISCOSITY_HIJ = {
    (0, 0): 5.20094e-1, (1, 0): 8.50895e-2,
    (2, 0): -1.08374, (3, 0): -2.89555e-1,
    (0, 1): 2.22531e-1, (1, 1): 9.99115e-1,
    (2, 1): 1.88797, (3, 1): 1.26613, (5, 1): 1.20573e-1,
    (0, 2): -2.81378e-1, (1, 2): -9.06851e-1,
    (2, 2): -7.72479e-1, (3, 2): -4.89837e-1,
    (4, 2): -2.57040e-1, (0, 3): 1.61913e-1,
    (1, 3): 2.57399e-1, (0, 4): -3.25372e-2,
    (3, 4): 6.98452e-2, (4, 5): 8.72102e-3,
    (3, 6): -4.35673e-3, (5, 6): -5.93264e-4,
}

IAPWS11_THERMAL_L0 = np.array([
    2.443221e-3, 1.323095e-2, 6.770357e-3,
    -3.454586e-3, 4.096266e-4,
], dtype=float)
IAPWS11_THERMAL_LIJ = np.array([
    [1.60397357, -0.646013523, 0.111443906, 0.102997357,
     -0.0504123634, 0.00609859258],
    [2.33771842, -2.78843778, 1.53616167, -0.463045512,
     0.0832827019, -0.00719201245],
    [2.19650529, -4.54580785, 3.55777244, -1.40944978,
     0.275418278, -0.0205938816],
    [-1.21051378, 1.60812989, -0.621178141, 0.0716373224,
     0.0, 0.0],
    [-2.7203370, 4.57586331, -3.18369245, 1.1168348,
     -0.19268305, 0.0129138427],
], dtype=float)
WATER_TC_K = 647.096
WATER_RHOC_KG_M3 = 322.0


def iapws_if97_region1_properties(temperature_K, pressure_Pa):
    """Return IF97 Region-1 v/rho/h/cp from the 34-term Gibbs equation."""
    temperature_K = float(temperature_K)
    pressure_Pa = float(pressure_Pa)
    if not (273.15 <= temperature_K <= 623.15):
        raise ValueError("IF97 Region 1 temperature range is 273.15..623.15 K")
    if not (611.213 <= pressure_Pa <= 100.0e6):
        raise ValueError("IF97 Region 1 pressure range is 611.213 Pa..100 MPa")
    pi = pressure_Pa / IF97_R1_PSTAR_PA
    tau = IF97_R1_TSTAR_K / temperature_K
    x = 7.1 - pi
    y = tau - 1.222
    gamma_pi = 0.0
    gamma_tau = 0.0
    gamma_tautau = 0.0
    for i, j, n in zip(IF97_R1_I, IF97_R1_J, IF97_R1_N):
        if i:
            gamma_pi -= n * i * x ** (i - 1) * y ** j
        if j:
            gamma_tau += n * j * x ** i * y ** (j - 1)
            gamma_tautau += n * j * (j - 1) * x ** i * y ** (j - 2)
    specific_volume = (
        IF97_R1_R_J_KG_K * temperature_K / pressure_Pa * pi * gamma_pi)
    density = 1.0 / specific_volume
    enthalpy = IF97_R1_R_J_KG_K * temperature_K * tau * gamma_tau
    cp = -IF97_R1_R_J_KG_K * tau * tau * gamma_tautau
    if not all(math.isfinite(value) and value > 0.0
               for value in (specific_volume, density, cp)):
        raise ValueError("IF97 Region 1 returned a nonphysical property")
    return {
        "temperature_K": temperature_K,
        "pressure_Pa": pressure_Pa,
        "specific_volume_m3_kg": specific_volume,
        "density_kg_m3": density,
        "enthalpy_J_kg": enthalpy,
        "cp_J_kgK": cp,
        "region": "IF97_REGION_1",
    }


def iapws_2008_viscosity_Pa_s(temperature_K, density_kg_m3):
    """IAPWS-2008 viscosity with the standard industrial mu2=1 branch."""
    tbar = float(temperature_K) / WATER_TC_K
    rhobar = float(density_kg_m3) / WATER_RHOC_KG_M3
    if not (0.0 < tbar <= 1173.15 / WATER_TC_K and rhobar >= 0.0):
        raise ValueError("IAPWS-2008 viscosity state is outside its range")
    mu0 = 100.0 * math.sqrt(tbar) / sum(
        value / tbar ** index
        for index, value in enumerate(IAPWS08_VISCOSITY_H0))
    delta_t = 1.0 / tbar - 1.0
    delta_rho = rhobar - 1.0
    exponent = rhobar * sum(
        value * delta_t ** i * delta_rho ** j
        for (i, j), value in IAPWS08_VISCOSITY_HIJ.items())
    # The critical-enhancement factor mu2 is set to one as explicitly
    # permitted for industrial use outside the small near-critical region.
    return mu0 * math.exp(exponent) * 1.0e-6


def iapws_2011_conductivity_W_mK(temperature_K, density_kg_m3):
    """IAPWS-2011 conductivity outside the negligible critical enhancement."""
    tbar = float(temperature_K) / WATER_TC_K
    rhobar = float(density_kg_m3) / WATER_RHOC_KG_M3
    if not (0.0 < tbar <= 1173.15 / WATER_TC_K and rhobar >= 0.0):
        raise ValueError("IAPWS-2011 conductivity state is outside its range")
    lambda0 = math.sqrt(tbar) / sum(
        value / tbar ** index
        for index, value in enumerate(IAPWS11_THERMAL_L0))
    delta_t = 1.0 / tbar - 1.0
    delta_rho = rhobar - 1.0
    exponent = rhobar * sum(
        IAPWS11_THERMAL_LIJ[i, j] * delta_t ** i * delta_rho ** j
        for i in range(IAPWS11_THERMAL_LIJ.shape[0])
        for j in range(IAPWS11_THERMAL_LIJ.shape[1])
    )
    # IAPWS-2011 permits lambda2=0 outside the near-critical region.  The
    # coolant screen stays below 623.15 K, so that industrial branch applies.
    # The IAPWS-2011 coefficient normalization produces mW m^-1 K^-1.
    return lambda0 * math.exp(exponent) * 1.0e-3


def water_properties_TP(temperature_K, pressure_Pa):
    """Return rho, mu, k, cp at a T/P state using the declared standards."""
    thermo = iapws_if97_region1_properties(temperature_K, pressure_Pa)
    rho = thermo["density_kg_m3"]
    return {
        **thermo,
        "dynamic_viscosity_Pa_s": iapws_2008_viscosity_Pa_s(
            temperature_K, rho),
        "conductivity_W_mK": iapws_2011_conductivity_W_mK(
            temperature_K, rho),
        "property_standards": {
            "thermodynamics": "IAPWS-IF97 Region 1",
            "viscosity": "IAPWS 2008 industrial branch (mu2=1)",
            "thermal_conductivity": "IAPWS 2011 industrial branch (lambda2=0)",
        },
    }


def verify_water_property_correlations():
    """Verify official release checkpoints before using properties in a run."""
    if not hasattr(verify_water_property_correlations, "result"):
        checks = []
        region1_cases = (
            (300.0, 3.0e6, 0.100215168e-2, 115.331273e3, 4.17301218e3),
            (300.0, 80.0e6, 0.971180894e-3, 184.142828e3, 4.01008987e3),
            (500.0, 3.0e6, 0.120241800e-2, 975.542239e3, 4.65580682e3),
        )
        for temperature, pressure, v_ref, h_ref, cp_ref in region1_cases:
            props = iapws_if97_region1_properties(temperature, pressure)
            checks.extend([
                math.isclose(props["specific_volume_m3_kg"], v_ref,
                             rel_tol=2.0e-7),
                math.isclose(props["enthalpy_J_kg"], h_ref, rel_tol=2.0e-7),
                math.isclose(props["cp_J_kgK"], cp_ref, rel_tol=2.0e-7),
            ])
        viscosity_cases = (
            (298.15, 998.0, 889.735100e-6),
            (373.15, 1000.0, 307.883622e-6),
            (433.15, 1000.0, 217.685358e-6),
        )
        for temperature, density, reference in viscosity_cases:
            checks.append(math.isclose(
                iapws_2008_viscosity_Pa_s(temperature, density),
                reference, rel_tol=2.0e-7))
        conductivity_cases = (
            (298.15, 0.0, 18.4341883e-3),
            (298.15, 998.0, 607.712868e-3),
        )
        for temperature, density, reference in conductivity_cases:
            checks.append(math.isclose(
                iapws_2011_conductivity_W_mK(temperature, density),
                reference, rel_tol=2.0e-7))
        if not all(checks):
            raise RuntimeError(
                "IAPWS property implementation failed official checkpoints")
        verify_water_property_correlations.result = {
            "status": "IAPWS_CORRELATIONS_VERIFIED",
            "check_count": len(checks),
            "sources": dict(WATER_PROPERTY_SOURCES),
        }
    return verify_water_property_correlations.result

class InputBlockedError(RuntimeError):
    """Raised when required inputs cannot be verified without substitution."""


def input_blocked(reason):
    raise InputBlockedError("INPUT_BLOCKED: %s" % reason)


def json_safe(value):
    """Convert NumPy values and non-finite floats into strict JSON values."""
    if isinstance(value, dict):
        return {str(k): json_safe(v) for k, v in value.items()}
    if isinstance(value, (list, tuple)):
        return [json_safe(v) for v in value]
    if isinstance(value, np.ndarray):
        return [json_safe(v) for v in value.tolist()]
    if isinstance(value, (np.integer, np.floating, np.bool_)):
        value = value.item()
    if isinstance(value, float):
        return value if math.isfinite(value) else None
    return value


def write_json(path, value):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(json_safe(value), indent=2, allow_nan=False) + "\n",
                    encoding="utf-8")


def sha256(path):
    h = hashlib.sha256()
    with Path(path).open("rb") as handle:
        for block in iter(lambda: handle.read(1024 * 1024), b""):
            h.update(block)
    return h.hexdigest()


def utc_now():
    return time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime())


def read_json(path, label=None):
    path = Path(path)
    label = label or str(path)
    if not path.is_file():
        input_blocked("required JSON is missing: %s" % label)
    try:
        value = json.loads(path.read_text(encoding="utf-8"))
    except (OSError, ValueError, TypeError) as exc:
        input_blocked("invalid JSON in %s: %s" % (label, exc))
    if not isinstance(value, dict):
        input_blocked("JSON root must be an object: %s" % label)
    return value


def _close(value, expected, tolerance=1.0e-12):
    try:
        return math.isclose(float(value), float(expected), rel_tol=0.0,
                            abs_tol=tolerance)
    except (TypeError, ValueError):
        return False


def audit_input_root(root):
    """Verify required files, declared hashes, model identity, and corrections."""
    root = Path(root).expanduser().resolve()
    for relative in REQUIRED_INPUT_FILES:
        if not (root / relative).is_file():
            input_blocked("required input is missing: %s" % relative)

    manifest_path = root / "dataset_manifest.json"
    manifest = read_json(manifest_path, "dataset_manifest.json")
    if manifest.get("model_version") != MODEL_VERSION:
        input_blocked(
            "dataset model_version mismatch: expected %s, found %r"
            % (MODEL_VERSION, manifest.get("model_version")))
    declared = manifest.get("files")
    if not isinstance(declared, dict) or not declared:
        input_blocked("dataset_manifest.json has no file-hash map")
    for relative in REQUIRED_INPUT_FILES:
        if relative == "dataset_manifest.json":
            continue
        if relative not in declared:
            input_blocked("manifest omits required file: %s" % relative)
    verified = {}
    for relative, expected_hash in sorted(declared.items()):
        candidate = (root / str(relative)).resolve()
        if candidate != root and root not in candidate.parents:
            input_blocked("manifest path escapes input root: %s" % relative)
        if not candidate.is_file():
            input_blocked("manifest-declared file is missing: %s" % relative)
        actual_hash = sha256(candidate)
        if actual_hash != str(expected_hash):
            input_blocked(
                "SHA-256 mismatch for %s: expected %s, found %s"
                % (relative, expected_hash, actual_hash))
        verified[str(relative)] = actual_hash

    mhd_input = read_json(root / "MHD" / "mhd_input.json",
                          "MHD/mhd_input.json")
    parameters = mhd_input.get("screening_parameters")
    if not isinstance(parameters, dict):
        input_blocked("MHD/mhd_input.json lacks screening_parameters")
    if not _close(parameters.get("tau_E_screening_target_s"),
                  EXPECTED_ACTIVE_TAU_E_S):
        input_blocked("active tau-E target must be 2.70 s")
    reference = parameters.get("reference_condition_correction")
    if not isinstance(reference, dict):
        input_blocked("reference_condition_correction is missing")
    if not _close(reference.get("tau_E_s"), EXPECTED_REFERENCE_TAU_E_S):
        input_blocked("reference-condition tau-E must be 1.36 s")
    if not _close(reference.get("self_heating_ratio"),
                  EXPECTED_REFERENCE_SELF_HEATING_RATIO):
        input_blocked("reference self-heating ratio must be 0.63")
    if reference.get("use_in_current_profile_prediction") is not False:
        input_blocked("the 63% reference result must not replace the active profile")
    routing = parameters.get("secondary_heat_removal_fraction_by_scenario")
    if not isinstance(routing, dict) or not _close(
            routing.get("full_incident_stress"),
            EXPECTED_FULL_INCIDENT_SECONDARY_FRACTION):
        input_blocked(
            "full_incident_stress secondary heat routing must be 0.50")
    reconciliation = read_json(
        root / "MHD" / "cad_kaggle_reconciliation.json",
        "MHD/cad_kaggle_reconciliation.json")
    cad_a2 = reconciliation.get("cad_a2_model_derived")
    if not isinstance(cad_a2, dict):
        input_blocked("CAD-A2 model-derived coil closure is missing")
    if cad_a2.get("turns_per_coil") != 100:
        input_blocked("CAD-A2 reference turn count must be the recovered 100-turn value")
    if cad_a2.get("coil_count") != 16:
        input_blocked("CAD-A2 TF coil count must be 16")
    property_sources = read_json(
        root / "MHD" / "water_property_sources.json",
        "MHD/water_property_sources.json")
    if property_sources.get("status") != "VERIFIED_OFFICIAL_IAPWS_RELEASES":
        input_blocked("water-property source manifest is not verified")
    expected_standards = {
        "thermodynamics": "IAPWS-IF97 Region 1",
        "viscosity": "IAPWS 2008",
        "thermal_conductivity": "IAPWS 2011",
    }
    if property_sources.get("standards") != expected_standards:
        input_blocked("water-property source manifest does not name the required standards")
    return {
        "passed": True,
        "root": str(root),
        "manifest_sha256": sha256(manifest_path),
        "verified_file_count": len(verified),
        "verified_files": verified,
        "model_version": manifest.get("model_version"),
        "cad_a2_turns_per_coil": cad_a2.get("turns_per_coil"),
        "water_property_sources_status": property_sources.get("status"),
    }


def discover_input_root(explicit=None):
    """Find one verified provisional_inputs directory without guessing."""
    if explicit:
        root = Path(explicit).expanduser().resolve()
        if root.name.upper() == "MHD" and root.parent.name == "provisional_inputs":
            root = root.parent
        audit_input_root(root)
        return root

    search_roots = []
    for value in (os.environ.get("TOMAHAWK_INPUT_ROOT"),
                  os.environ.get("TOMAHAWK_KAGGLE_INPUT_ROOT"),
                  "/kaggle/input",
                  os.environ.get("TOMAHAWK_KAGGLE_WORKING_ROOT"),
                  "/kaggle/working", str(Path.cwd())):
        if value:
            candidate = Path(value)
            if candidate.exists() and candidate not in search_roots:
                search_roots.append(candidate)
    candidates = []
    for search_root in search_roots:
        if (search_root / "MHD" / "mhd_profile_sweep.csv").is_file():
            candidates.append(search_root.resolve())
        try:
            matches = sorted(search_root.rglob("mhd_profile_sweep.csv"))
        except OSError:
            matches = []
        for match in matches:
            if match.parent.name == "MHD":
                candidates.append(match.parent.parent.resolve())
    candidates = list(dict.fromkeys(candidates))
    valid = []
    rejected = []
    for candidate in candidates:
        try:
            audit = audit_input_root(candidate)
            valid.append((candidate, audit))
        except InputBlockedError as exc:
            rejected.append("%s -> %s" % (candidate, exc))
    if not valid:
        detail = "; ".join(rejected[:4]) if rejected else "no candidates found"
        input_blocked("no verified provisional_inputs bundle: %s" % detail)
    fingerprints = {item[1]["manifest_sha256"] for item in valid}
    if len(fingerprints) != 1:
        input_blocked(
            "multiple different verified input bundles were found; set "
            "TOMAHAWK_INPUT_ROOT explicitly")
    return sorted(valid, key=lambda item: (len(str(item[0])), str(item[0])))[0][0]


def load_profile(path):
    rows = []
    with Path(path).open(newline="", encoding="utf-8") as handle:
        for line_number, raw in enumerate(csv.DictReader(handle), start=2):
            try:
                rows.append({
                    "normalized_flux": float(raw["normalized_flux"]),
                    "pressure_Pa": float(raw["pressure_Pa"]),
                    "electron_temperature_keV": float(raw["electron_temperature_keV"]),
                    "ion_temperature_keV": float(raw["ion_temperature_keV"]),
                    "density_m3": float(raw["density_m3"]),
                    "value_status": raw.get("value_status", ""),
                })
            except (KeyError, TypeError, ValueError) as exc:
                input_blocked(
                    "malformed MHD profile row %d: %s" % (line_number, exc))
    if not rows:
        input_blocked("MHD profile CSV is empty: %s" % path)
    rows.sort(key=lambda item: item["normalized_flux"])
    return rows


def load_cfd_rows(path):
    rows = []
    if not path or not Path(path).is_file():
        input_blocked("CFD/CHT parameter sweep is missing: %s" % path)
    with Path(path).open(newline="", encoding="utf-8") as handle:
        for line_number, raw in enumerate(csv.DictReader(handle), start=2):
            try:
                rows.append({
                    "case_id": raw["case_id"],
                    "channel_count": int(raw["channel_count"]),
                    "hydraulic_diameter_m": float(raw["hydraulic_diameter_m"]),
                    "velocity_m_s": float(raw["velocity_m_s"]),
                    "allowed_bulk_rise_K": float(raw["allowed_bulk_rise_K"]),
                    "heat_flux_scenario": raw["heat_flux_scenario"],
                    "local_heat_flux_W_m2": float(raw["local_heat_flux_W_m2"]),
                    "pressure_Pa": float(raw["pressure_Pa"]),
                    "inlet_temperature_K": float(raw["inlet_temperature_K"]),
                    "screening_Darcy_pressure_drop_Pa": float(
                        raw["screening_Darcy_pressure_drop_Pa"]),
                    "pressure_drop_limit_Pa": float(raw["pressure_drop_limit_Pa"]),
                    "CuCrZr_temperature_limit_K": float(
                        raw["CuCrZr_temperature_limit_K"]),
                })
            except (KeyError, TypeError, ValueError) as exc:
                input_blocked(
                    "malformed CFD/CHT row %d: %s" % (line_number, exc))
    if not rows:
        input_blocked("CFD/CHT parameter sweep contains no usable rows")
    return rows


def load_bundle(input_root=None):
    root = discover_input_root(input_root)
    audit = audit_input_root(root)
    paths = {
        "mhd_input": root / "MHD" / "mhd_input.json",
        "profile": root / "MHD" / "mhd_profile_sweep.csv",
        "cfd_sweep": root / "CFD" / "cfd_cht_parameter_sweep.csv",
        "chf_input": root / "CHF" / "chf_input.json",
        "cad_a2_winding_envelope": root / "MHD" / "cad_a2_winding_envelope.csv",
        "cad_a2_coil_centerlines": root / "MHD" / "cad_a2_coil_centerlines.csv",
        "cad_a2_coil_currents": root / "MHD" / "cad_a2_coil_currents.csv",
        "cad_kaggle_reconciliation": root / "MHD" / "cad_kaggle_reconciliation.json",
        "water_property_sources": root / "MHD" / "water_property_sources.json",
        "dataset_manifest": root / "dataset_manifest.json",
    }
    files = {
        name: {
            "path": str(path.resolve()),
            "sha256": sha256(path),
            "present": True,
        }
        for name, path in paths.items()
    }
    profile = load_profile(paths["profile"])
    mhd_input = read_json(paths["mhd_input"], "MHD/mhd_input.json")
    cfd_rows = load_cfd_rows(paths["cfd_sweep"])
    chf_input = read_json(paths["chf_input"], "CHF/chf_input.json")
    cad_kaggle_reconciliation = read_json(
        paths["cad_kaggle_reconciliation"],
        "MHD/cad_kaggle_reconciliation.json")
    water_property_sources = read_json(
        paths["water_property_sources"],
        "MHD/water_property_sources.json")
    return {
        "input_root": str(root),
        "files": files,
        "profile": profile,
        "mhd_input": mhd_input,
        "cfd_rows": cfd_rows,
        "chf_input": chf_input,
        "cad_a2_winding_envelope": paths["cad_a2_winding_envelope"],
        "cad_a2_coil_centerlines": paths["cad_a2_coil_centerlines"],
        "cad_a2_coil_currents": paths["cad_a2_coil_currents"],
        "cad_kaggle_reconciliation": cad_kaggle_reconciliation,
        "water_property_sources": water_property_sources,
        "source_status": "BUNDLE_INPUTS_VERIFIED",
        "input_audit": audit,
    }


def profile_arrays(profile):
    return {
        key: np.asarray([row[key] for row in profile], dtype=float)
        for key in ("normalized_flux", "pressure_Pa",
                    "electron_temperature_keV", "ion_temperature_keV",
                    "density_m3")
    }


def validate_profile(profile):
    arrays = profile_arrays(profile)
    rho = arrays["normalized_flux"]
    density = arrays["density_m3"]
    te = arrays["electron_temperature_keV"]
    ti = arrays["ion_temperature_keV"]
    supplied_p = arrays["pressure_Pa"]
    calculated_p = density * (te + ti) * KEV_TO_J
    relative = np.abs(supplied_p - calculated_p) / np.maximum(np.abs(calculated_p), 1.0)
    gates = {
        "finite_values": bool(np.all(np.isfinite(np.column_stack(
            (rho, density, te, ti, supplied_p))))),
        "flux_strictly_increasing": bool(np.all(np.diff(rho) > 0.0)),
        "flux_bounded_0_1": bool(np.min(rho) >= 0.0 and np.max(rho) <= 1.0),
        "positive_density_and_temperature": bool(
            np.all(density > 0.0) and np.all(te > 0.0) and np.all(ti > 0.0)),
        "pressure_closure_pass": bool(np.max(relative) <= 2.0e-5),
    }
    return {
        "rows": int(len(profile)),
        "rho_min": float(np.min(rho)),
        "rho_max": float(np.max(rho)),
        "central_density_m3": float(density[0]),
        "central_electron_temperature_keV": float(te[0]),
        "central_ion_temperature_keV": float(ti[0]),
        "central_pressure_Pa_supplied": float(supplied_p[0]),
        "central_pressure_Pa_reconstructed": float(calculated_p[0]),
        "maximum_pressure_closure_relative_error": float(np.max(relative)),
        "gates": gates,
        "profile_consistency_pass": bool(all(gates.values())),
        "density_definition": "total quasi-neutral DT ion/electron density; nD=nT=n_total/2",
        "status": "PROVISIONAL_PROFILE_CONSISTENCY_CHECK",
    }, arrays


def dt_reactivity(T_keV):
    """Bosch-Hale D-T reactivity in m^3/s for a Maxwellian ion temperature."""
    T = np.clip(np.asarray(T_keV, dtype=float), 0.2, 100.0)
    p = BH_DT
    numerator = T * (p["C2"] + T * (p["C4"] + T * p["C6"]))
    denominator = 1.0 + T * (p["C3"] + T * (p["C5"] + T * p["C7"]))
    theta = T / (1.0 - numerator / denominator)
    xi = (p["B_G_sqrt_keV"] ** 2 / (4.0 * theta)) ** (1.0 / 3.0)
    cm3_per_s = (p["C1"] * theta
                  * np.sqrt(xi / (p["mrc2_keV"] * T ** 3))
                  * np.exp(-3.0 * xi))
    return 1.0e-6 * cm3_per_s


def make_grid(config, nr=161, nz=161):
    R0, a, kappa = config["R0_m"], config["a_m"], config["kappa"]
    R = np.linspace(R0 - 1.10 * a, R0 + 1.10 * a, int(nr))
    Z = np.linspace(-1.10 * kappa * a, 1.10 * kappa * a, int(nz))
    RR, ZZ = np.meshgrid(R, Z, indexing="xy")
    dR, dZ = float(R[1] - R[0]), float(Z[1] - Z[0])
    rho2 = ((RR - R0) / a) ** 2 + (ZZ / (kappa * a)) ** 2
    mask = rho2 <= 1.0
    return {
        "R": R, "Z": Z, "RR": RR, "ZZ": ZZ, "dR": dR, "dZ": dZ,
        "rho": np.sqrt(np.maximum(rho2, 0.0)), "mask": mask,
    }


def delta_star(psi, grid):
    """Finite-difference Delta-star operator with zero Dirichlet exterior."""
    RR, mask = grid["RR"], grid["mask"]
    dR, dZ = grid["dR"], grid["dZ"]
    east = np.zeros_like(psi)
    west = np.zeros_like(psi)
    north = np.zeros_like(psi)
    south = np.zeros_like(psi)
    east[:, :-1] = np.where(mask[:, 1:], psi[:, 1:], 0.0)
    west[:, 1:] = np.where(mask[:, :-1], psi[:, :-1], 0.0)
    north[:-1, :] = np.where(mask[1:, :], psi[1:, :], 0.0)
    south[1:, :] = np.where(mask[:-1, :], psi[:-1, :], 0.0)
    ae = 1.0 / dR ** 2 - 1.0 / (2.0 * RR * dR)
    aw = 1.0 / dR ** 2 + 1.0 / (2.0 * RR * dR)
    ap = -2.0 / dR ** 2 - 2.0 / dZ ** 2
    return ae * east + aw * west + north / dZ ** 2 + south / dZ ** 2 + ap * psi


def build_sparse_operator(grid):
    """Build the linear fixed-boundary Delta-star matrix."""
    from scipy import sparse

    RR, mask = grid["RR"], grid["mask"]
    dR, dZ = grid["dR"], grid["dZ"]
    nz, nr = mask.shape
    index = -np.ones(mask.shape, dtype=int)
    index[mask] = np.arange(int(mask.sum()))
    ae = 1.0 / dR ** 2 - 1.0 / (2.0 * RR * dR)
    aw = 1.0 / dR ** 2 + 1.0 / (2.0 * RR * dR)
    an = 1.0 / dZ ** 2
    ap = -2.0 / dR ** 2 - 2.0 / dZ ** 2
    row, col, data = [], [], []
    for i, j in zip(*np.nonzero(mask)):
        current = int(index[i, j])
        row.append(current)
        col.append(current)
        data.append(float(ap))
        if j + 1 < nr and mask[i, j + 1]:
            row.append(current)
            col.append(int(index[i, j + 1]))
            data.append(float(ae[i, j]))
        if j - 1 >= 0 and mask[i, j - 1]:
            row.append(current)
            col.append(int(index[i, j - 1]))
            data.append(float(aw[i, j]))
        if i + 1 < nz and mask[i + 1, j]:
            row.append(current)
            col.append(int(index[i + 1, j]))
            data.append(float(an))
        if i - 1 >= 0 and mask[i - 1, j]:
            row.append(current)
            col.append(int(index[i - 1, j]))
            data.append(float(an))
    matrix = sparse.coo_matrix((data, (row, col)),
                               shape=(int(mask.sum()), int(mask.sum()))).tocsr()
    return matrix, index


def solve_gs_sor(grid, source, max_iters):
    """Portable fallback for the fixed-source linear equation."""
    RR, mask = grid["RR"], grid["mask"]
    dR, dZ = grid["dR"], grid["dZ"]
    ae = 1.0 / dR ** 2 - 1.0 / (2.0 * RR * dR)
    aw = 1.0 / dR ** 2 + 1.0 / (2.0 * RR * dR)
    ap = -2.0 / dR ** 2 - 2.0 / dZ ** 2
    checker = (np.indices(mask.shape).sum(axis=0) % 2) == 0
    psi = np.zeros_like(source)
    relaxation = 1.65
    scale = max(float(np.max(np.abs(source[mask]))), 1.0)
    relative = math.inf
    used = int(max_iters)
    for iteration in range(1, int(max_iters) + 1):
        east = np.zeros_like(psi)
        west = np.zeros_like(psi)
        north = np.zeros_like(psi)
        south = np.zeros_like(psi)
        east[:, :-1] = np.where(mask[:, 1:], psi[:, 1:], 0.0)
        west[:, 1:] = np.where(mask[:, :-1], psi[:, :-1], 0.0)
        north[:-1, :] = np.where(mask[1:, :], psi[1:, :], 0.0)
        south[1:, :] = np.where(mask[:-1, :], psi[:-1, :], 0.0)
        trial = (source - ae * east - aw * west
                 - (north + south) / dZ ** 2) / ap
        for parity in (True, False):
            active = mask & (checker == parity)
            psi[active] = ((1.0 - relaxation) * psi[active]
                           + relaxation * trial[active])
        psi[~mask] = 0.0
        if iteration == 1 or iteration % 25 == 0:
            residual = delta_star(psi, grid) - source
            relative = float(np.max(np.abs(residual[mask])) / scale)
            if relative <= 1.0e-6:
                used = iteration
                break
    residual = delta_star(psi, grid) - source
    relative = float(np.max(np.abs(residual[mask])) / scale)
    return psi, used, relative, "numpy_sor"


def solve_fixed_boundary_gs(config, target_pressure_pa, nr=161, nz=161,
                            max_iters=12000):
    """Solve Delta-star psi = -mu0 R^2 dp/dpsi for a fixed circular boundary."""
    grid = make_grid(config, nr=nr, nz=nz)
    RR, mask = grid["RR"], grid["mask"]
    unit_source = np.where(mask, -MU0 * RR ** 2, 0.0)
    method = "numpy_sor"
    iterations = int(max_iters)
    if target_pressure_pa <= 0.0:
        raise ValueError("The target central pressure must be positive.")

    try:
        matrix, index = build_sparse_operator(grid)
        from scipy.sparse.linalg import spsolve
        unit_flat = spsolve(matrix, unit_source[mask])
        unit_psi = np.zeros_like(RR)
        unit_psi[mask] = unit_flat
        method = "scipy_sparse_direct"
        iterations = 1
        solver_residual = float(np.max(
            np.abs(delta_star(unit_psi, grid)[mask] - unit_source[mask]))
            / max(float(np.max(np.abs(unit_source[mask]))), 1.0))
    except (ImportError, ModuleNotFoundError, RuntimeError, ValueError):
        unit_psi, iterations, solver_residual, method = solve_gs_sor(
            grid, unit_source, max_iters)

    psi_axis_unit = float(np.max(unit_psi[mask]))
    if not math.isfinite(psi_axis_unit) or psi_axis_unit <= 0.0:
        raise RuntimeError("Fixed-boundary GS solve returned a non-positive axis flux.")
    # The source is proportional to dp/dpsi, so the solved psi is also
    # proportional to dp/dpsi.  Because p = (dp/dpsi)*psi, the axis pressure
    # is quadratic in this calibration factor.
    dp_dpsi = math.sqrt(target_pressure_pa / psi_axis_unit)
    psi = dp_dpsi * unit_psi
    source = dp_dpsi * unit_source
    residual = delta_star(psi, grid) - source
    relative = float(np.max(np.abs(residual[mask]))
                    / max(float(np.max(np.abs(source[mask]))), 1.0))
    return grid, psi, source, {
        "method": method,
        "iterations": iterations,
        "dp_dpsi_Pa_Wb": float(dp_dpsi),
        "residual_relative": relative,
        "unit_solve_residual_relative": solver_residual,
        "target_pressure_Pa": float(target_pressure_pa),
    }


def equilibrium_metrics(config, grid, psi, source, gs_info):
    RR, mask, rho = grid["RR"], grid["mask"], grid["rho"]
    dR, dZ = grid["dR"], grid["dZ"]
    p_axis = float(np.max(psi[mask]) * gs_info["dp_dpsi_Pa_Wb"])
    p_eq = np.where(mask, np.maximum(psi * gs_info["dp_dpsi_Pa_Wb"], 0.0), 0.0)
    dpsi_dz, dpsi_dr = np.gradient(psi, dZ, dR, edge_order=2)
    br = np.where(mask, -dpsi_dz / RR, 0.0)
    bz = np.where(mask, dpsi_dr / RR, 0.0)
    f_edge = config["R0_m"] * config["B0_plasma_T"]
    bphi = np.where(mask, f_edge / RR, 0.0)
    jphi = np.where(mask, RR * gs_info["dp_dpsi_Pa_Wb"], 0.0)
    dA = dR * dZ
    volume = float(np.sum((2.0 * math.pi * RR)[mask]) * dA)
    ip = float(np.sum(jphi[mask]) * dA)
    core = mask & (rho <= 0.90)
    divb = (np.gradient(RR * br, dR, axis=1, edge_order=2) / RR
            + np.gradient(bz, dZ, axis=0, edge_order=2))
    field_scale = max(float(np.max(np.abs(br[core]))),
                      float(np.max(np.abs(bz[core]))), 1.0) / config["a_m"]
    divb_relative = float(np.max(np.abs(divb[core])) / field_scale)
    ip95 = float(np.sum(jphi[mask & (rho <= 0.95)]) * dA)
    q95 = (2.0 * math.pi * (0.95 * config["a_m"]) ** 2
           * config["B0_plasma_T"]
           / (MU0 * config["R0_m"] * max(abs(ip95), 1.0)))
    p_volume = float(np.sum(p_eq[mask] * (2.0 * math.pi * RR[mask])) * dA
                     / max(volume, 1.0))
    beta_axis = 2.0 * MU0 * p_axis / config["B0_plasma_T"] ** 2
    beta_volume = 2.0 * MU0 * p_volume / config["B0_plasma_T"] ** 2
    beta_n = 100.0 * beta_volume * config["a_m"] * config["B0_plasma_T"] / max(
        abs(ip) / 1.0e6, 1.0e-12)
    shells = np.linspace(0.05, 0.95, 19)
    shell_p = np.asarray([
        np.nanmean(p_eq[(rho >= value - 0.025) & (rho < value + 0.025)])
        for value in shells
    ])
    finite_shell = shell_p[np.isfinite(shell_p)]
    monotonic = bool(len(finite_shell) > 1
                     and np.all(np.diff(finite_shell) <= 1.0e-6 * max(p_axis, 1.0)))
    residual_relative = float(gs_info["residual_relative"])
    gates = {
        "GS_residual_pass": residual_relative <= config["GS_residual_tolerance"],
        "divB_pass": divb_relative <= config["divB_relative_tolerance"],
        "positive_pressure_pass": bool(p_axis > 0.0 and np.min(p_eq[mask]) >= -1.0e-9),
        "pressure_monotonic_pass": monotonic,
        "q95_proxy_pass": bool(q95 >= config["q95_proxy_min"]),
        "beta_N_proxy_pass": bool(beta_n <= config["beta_N_proxy_max"]),
        "beta_axis_proxy_pass": bool(
            100.0 * beta_axis <= config["beta_axis_proxy_max_percent"]),
    }
    gates["equilibrium_numerical_pass"] = bool(all(
        gates[name] for name in ("GS_residual_pass", "divB_pass",
                                 "positive_pressure_pass")))
    gates["ideal_MHD_proxy_pass"] = bool(all(
        gates[name] for name in ("pressure_monotonic_pass", "q95_proxy_pass",
                                 "beta_N_proxy_pass", "beta_axis_proxy_pass")))
    metrics = {
        "solver_method": gs_info["method"],
        "gs_iterations": gs_info["iterations"],
        "gs_residual_relative": residual_relative,
        "unit_solve_residual_relative": gs_info["unit_solve_residual_relative"],
        "divB_relative": divb_relative,
        "psi_axis_Wb": float(np.max(psi[mask])),
        "p_axis_MPa": p_axis / 1.0e6,
        "p_volume_average_MPa": p_volume / 1.0e6,
        "I_p_solved_MA": ip / 1.0e6,
        "I_p_95_MA": ip95 / 1.0e6,
        "q95_cylindrical_proxy": q95,
        "beta_axis_percent": 100.0 * beta_axis,
        "beta_volume_percent": 100.0 * beta_volume,
        "beta_N_proxy": beta_n,
        "grid_toroidal_volume_m3": volume,
        "CAD_volume_m3": config["CAD_volume_m3"],
        "grid_to_CAD_volume_ratio": volume / config["CAD_volume_m3"],
        "magnetic_pressure_axis_MPa": config["B0_plasma_T"] ** 2 / (2.0 * MU0) / 1e6,
        "magnetic_pressure_coil_MPa": config["Bpeak_coil_T"] ** 2 / (2.0 * MU0) / 1e6,
        "gates": gates,
        "status": "FIXED_BOUNDARY_AXISYMMETRIC_SCREEN_NOT_3D_MHD",
    }
    fields = {
        "psi": psi, "pressure": p_eq, "BR": br, "BZ": bz, "BPHI": bphi,
        "JPHI": jphi,
    }
    return metrics, fields


def weighted_profile_values(profile, arrays, quadrature_order=128):
    nodes, weights = np.polynomial.legendre.leggauss(quadrature_order)
    rq = 0.5 * (nodes + 1.0)
    volume_weights = 0.5 * weights * (nodes + 1.0)
    values = {
        key: np.interp(rq, arrays["normalized_flux"], arrays[key])
        for key in ("pressure_Pa", "electron_temperature_keV",
                    "ion_temperature_keV", "density_m3")
    }
    return rq, volume_weights, values


def confinement_metrics(config, profile, arrays):
    rq, weights, values = weighted_profile_values(profile, arrays)
    density = values["density_m3"]
    n_d = density / 2.0
    n_t = density / 2.0
    n_e = density
    n_i = density
    te = values["electron_temperature_keV"]
    ti = values["ion_temperature_keV"]
    reactivity = dt_reactivity(ti)
    reaction_rate = config["CAD_volume_m3"] * float(np.sum(
        weights * n_d * n_t * reactivity))
    p_fusion = reaction_rate * E_FUSION_J
    p_alpha = p_fusion * ALPHA_FRACTION
    p_neutron = p_fusion * NEUTRON_FRACTION
    p_alpha_deposited = p_alpha * config["alpha_deposition_fraction"]
    # The coefficient is expressed for electron temperature in keV.  Do not
    # apply an extra sqrt(1000) conversion here; that was the source of the
    # earlier overestimated bremsstrahlung loss and failed smoke gate.
    p_brem_density = 5.35e-37 * config["Z_eff"] * n_e ** 2 * np.sqrt(te)
    p_brem = config["CAD_volume_m3"] * float(np.sum(weights * p_brem_density))
    pressure_reconstructed = n_i * (ti + te) * KEV_TO_J
    p_avg = float(np.sum(weights * pressure_reconstructed))
    w_thermal = 1.5 * config["CAD_volume_m3"] * p_avg
    p_transport = w_thermal / config["tau_E_s"]
    modeled_losses = p_transport + p_brem
    p_aux = config["auxiliary_absorbed_MW"] * 1.0e6
    alpha_ratio = p_alpha_deposited / max(modeled_losses, 1.0)
    if p_alpha_deposited > p_brem:
        tau_required = w_thermal / (p_alpha_deposited - p_brem)
        tau_margin = config["tau_E_s"] / tau_required - 1.0
    else:
        tau_required = math.inf
        tau_margin = -math.inf
    n_avg = float(np.sum(weights * density))
    ti_avg = float(np.sum(weights * ti))
    te_avg = float(np.sum(weights * te))
    reference_ratio = config.get("reference_self_heating_ratio")
    reference_tau_e = float(config.get(
        "reference_condition_tau_E_s",
        DEFAULTS["reference_condition_tau_E_s"]))
    reference_transport = w_thermal / max(reference_tau_e, 1.0e-30)
    reference_modeled_losses = reference_transport + p_brem
    reference_ratio_computed = p_alpha_deposited / max(
        reference_modeled_losses, 1.0)
    if reference_ratio is None:
        reference_delta = None
        reference_reconciled = None
        reference_status = "NOT_SUPPLIED"
    else:
        reference_delta = float(reference_ratio - reference_ratio_computed)
        reference_reconciled = bool(
            abs(reference_delta)
            <= config["reference_self_heating_reconciliation_tolerance"])
        reference_status = (
            "RECONCILED_TO_CURRENT_PROFILE"
            if reference_reconciled else
            "BLOCKED_UNMAPPED_REFERENCE_CORRECTION")
    metrics = {
        "fusion_GW_module": p_fusion / 1.0e9,
        "alpha_GW_module": p_alpha / 1.0e9,
        "alpha_deposited_GW_module": p_alpha_deposited / 1.0e9,
        "neutron_GW_module": p_neutron / 1.0e9,
        "bremsstrahlung_GW_module": p_brem / 1.0e9,
        "bremsstrahlung_equation": (
            "P_brem = 5.35e-37 * Z_eff * n_e^2 * sqrt(T_e_keV)"),
        "bremsstrahlung_temperature_units": "keV",
        "transport_loss_GW_module": p_transport / 1.0e9,
        "modeled_total_loss_GW_module": modeled_losses / 1.0e9,
        "thermal_energy_GJ_module": w_thermal / 1.0e9,
        "absorbed_auxiliary_MW_module": p_aux / 1.0e6,
        "Q_fusion_over_absorbed_auxiliary": p_fusion / max(p_aux, 1.0),
        "Q_fusion_over_modeled_losses": p_fusion / max(modeled_losses, 1.0),
        "alpha_self_heating_ratio": alpha_ratio,
        "tau_E_assumption_s": config["tau_E_s"],
        "active_tau_E_screening_target_s": config["tau_E_s"],
        "active_tau_E_screening_target_provenance": config.get(
            "active_tau_E_screening_target_provenance"),
        "tau_E_required_for_alpha_balance_s": tau_required,
        "tau_E_margin_ratio": tau_margin,
        "net_alpha_margin_GW_module": (
            p_alpha_deposited - p_brem - p_transport) / 1.0e9,
        "triple_product_nT_keV_s_m3": n_avg * ti_avg * config["tau_E_s"],
        "volume_weighted_density_m3": n_avg,
        "volume_weighted_Ti_keV": ti_avg,
        "volume_weighted_Te_keV": te_avg,
        "reactivity_at_volume_weighted_Ti_m3_s": float(dt_reactivity(ti_avg)),
        "reactivity_model": BH_DT["source"],
        "volume_weighting": "normalized-radius weight 2*rho; screening approximation",
        "self_heating_definition": (
            "deposited alpha power / (bremsstrahlung + W/tau_E)"),
        "reference_condition_self_heating_ratio": reference_ratio,
        "reference_condition_self_heating_ratio_computed": (
            reference_ratio_computed if reference_ratio is not None else None),
        "reference_condition_tau_E_s": reference_tau_e,
        "reference_condition_self_heating_ratio_delta": reference_delta,
        "reference_condition_self_heating_definition": config.get(
            "reference_self_heating_definition"),
        "reference_condition_self_heating_provenance": config.get(
            "reference_self_heating_provenance"),
        "reference_condition_self_heating_source_status": config.get(
            "reference_self_heating_source_status"),
        "reference_condition_self_heating_used_as_model_input": config.get(
            "reference_self_heating_used_as_model_input", False),
        "reference_condition_reconciliation_tolerance": config.get(
            "reference_self_heating_reconciliation_tolerance"),
        "reference_condition_reconciliation_status": reference_status,
        "gates": {
            "alpha_self_heating_pass": bool(alpha_ratio >= 1.0),
            "tau_E_margin_pass": bool(tau_margin >= config["tau_E_margin_min"]),
            "reference_self_heating_reconciliation_pass": reference_reconciled,
        },
        "status": "0D_DT_POWER_AND_CONFINEMENT_SCREEN_NOT_BURNING_PLASMA_VALIDATION",
    }
    return metrics


def darcy_pressure_drop(rho, velocity, diameter, length, roughness, dynamic_mu):
    reynolds = rho * velocity * diameter / dynamic_mu
    if reynolds <= 0.0:
        return 0.0, reynolds, 0.0
    friction = 0.25 / math.log10(
        roughness / (3.7 * diameter) + 5.74 / reynolds ** 0.9) ** 2
    dp = friction * (length / diameter) * rho * velocity ** 2 / 2.0
    return dp, reynolds, friction


# ---------------------------------------------------------------------------
# Real critical-heat-flux (CHF) screen, replacing the CuCrZr-wall-temperature
# -only proxy with an actual departure-from-nucleate-boiling assessment.
#
# Two pieces, each independently sourced and numerically verified against
# published checkpoints (see accompanying test output):
#
#   1. iapws_if97_tsat_K -- the EXACT IAPWS-IF97 Region-4 backward equation
#      Tsat(p) (Revised Release on the IAPWS Industrial Formulation 1997,
#      Table 34). Verified against published steam-table checkpoints:
#      1 MPa -> 179.9 C, 7 MPa -> 285.8 C, 10 MPa -> 311.1 C.
#
#   2. biasi_chf_W_m2 -- the Biasi (1967) CHF correlation, as tabulated in
#      e.g. Militello & Avramova, "A Methodology for CHF Prediction in VVER
#      Rod Bundles" (arXiv:2203.15048), eq. 41-44, cross-checked against a
#      second independent source (a 1989 JAEA/JNST technical report). Valid
#      range: D_h 0.003-0.0375 m, heated length 0.2-6.0 m, P 0.27-14 MPa,
#      mass flux G approx 100-6000 kg/(m^2 s). This screen flags rows
#      outside that envelope rather than silently extrapolating.
#
# Local thermodynamic equilibrium quality (Biasi's required input) is
# estimated from bulk subcooling using a simple single-phase energy-balance
# approximation, x_e = -cp*(Tsat-Tbulk)/hfg, with hfg estimated from the
# Watson correlation referenced to the well-known atmospheric value
# (Tb=373.15 K, hfg=2256.4 kJ/kg) and the critical temperature of water
# (Tc=647.096 K). This is a standard, widely used way to extend a single
# reference latent-heat value to other temperatures; it is NOT a full
# steam-table (IAPWS-95/IF97) enthalpy calculation.
# ---------------------------------------------------------------------------
IAPWS_IF97_REGION4_N = (
    1167.0521452767, -724213.16703206, -17.073846940092, 12020.82470247,
    -3232555.0322333, 14.91510861353, -4823.2657361591, 405113.40542057,
    -0.23855557567849, 650.17534844798)
IAPWS_IF97_REGION4_MIN_PA = 611.213
IAPWS_IF97_REGION4_MAX_PA = 22.064e6
WATER_TC_K = 647.096
WATSON_REFERENCE_TB_K = 373.15
WATSON_REFERENCE_HFG_J_KG = 2256.4e3
WATSON_EXPONENT = 0.38
BIASI_VALID_RANGE = {
    "hydraulic_diameter_m": (0.003, 0.0375),
    "heated_length_m": (0.2, 6.0),
    "pressure_Pa": (0.27e6, 14.0e6),
    "mass_flux_kg_m2s": (100.0, 6000.0),
}


def iapws_if97_tsat_K(pressure_pa):
    """Exact IAPWS-IF97 Region-4 backward equation, Tsat(p). See module
    notes above for source and verification. Raises ValueError outside the
    equation's declared validity range rather than extrapolating silently.
    """
    if not (IAPWS_IF97_REGION4_MIN_PA <= pressure_pa <= IAPWS_IF97_REGION4_MAX_PA):
        raise ValueError(
            "pressure %.6g Pa is outside the IAPWS-IF97 Region-4 Tsat "
            "validity range [%.6g, %.6g] Pa"
            % (pressure_pa, IAPWS_IF97_REGION4_MIN_PA, IAPWS_IF97_REGION4_MAX_PA))
    n = IAPWS_IF97_REGION4_N
    p_mpa = pressure_pa / 1.0e6
    beta = p_mpa ** 0.25
    e = beta ** 2 + n[2] * beta + n[5]
    f = n[0] * beta ** 2 + n[3] * beta + n[6]
    g = n[1] * beta ** 2 + n[4] * beta + n[7]
    d = 2.0 * g / (-f - math.sqrt(f ** 2 - 4.0 * e * g))
    return 0.5 * (n[9] + d - math.sqrt((n[9] + d) ** 2 - 4.0 * (n[8] + n[9] * d)))


def watson_latent_heat_J_kg(temperature_K):
    """Latent heat of vaporization via the Watson correlation, referenced
    to the standard atmospheric value. Screening-accuracy only."""
    t = min(max(temperature_K, 273.16), WATER_TC_K - 1.0e-6)
    ratio = (WATER_TC_K - t) / (WATER_TC_K - WATSON_REFERENCE_TB_K)
    return WATSON_REFERENCE_HFG_J_KG * ratio ** WATSON_EXPONENT


def local_equilibrium_quality(bulk_temperature_K, pressure_pa, cp_j_kgK):
    """Approximate thermodynamic equilibrium quality from bulk subcooling.
    Negative for subcooled liquid (the expected regime here). See module
    notes for the approximation and its limits."""
    tsat = iapws_if97_tsat_K(pressure_pa)
    hfg = watson_latent_heat_J_kg(tsat)
    return -cp_j_kgK * max(tsat - bulk_temperature_K, 0.0) / hfg, tsat, hfg


def biasi_chf_W_m2(pressure_pa, mass_flux_kg_m2s, hydraulic_diameter_m,
                    quality):
    """Biasi (1967) CHF correlation. Inputs in SI; converted internally to
    the correlation's native units (D in cm, G in g/(cm^2 s), P in atm).
    Returns (chf_W_m2, in_valid_range: bool)."""
    d_cm = hydraulic_diameter_m * 100.0
    g_g_cm2s = mass_flux_kg_m2s * 0.1  # kg/(m^2 s) -> g/(cm^2 s)
    p_atm = pressure_pa / 101325.0
    alpha = 0.6 if hydraulic_diameter_m < 0.01 else 0.4
    a = 0.7249 + 0.099 * p_atm * math.exp(-0.032 * p_atm)
    b = (-1.159 + 0.149 * p_atm * math.exp(-0.019 * p_atm)
         + 8.99 * p_atm / (10.0 + p_atm ** 2))
    low_branch = (1.883e4 / (d_cm ** alpha * g_g_cm2s ** (1.0 / 6.0))
                  * (a / g_g_cm2s ** (1.0 / 6.0) - quality))
    high_branch = (3.78e4 * b / (d_cm ** alpha * g_g_cm2s ** 0.6)
                   * (1.0 - quality))
    chf_kW_m2 = max(low_branch, high_branch)
    in_range = (
        BIASI_VALID_RANGE["hydraulic_diameter_m"][0] <= hydraulic_diameter_m
        <= BIASI_VALID_RANGE["hydraulic_diameter_m"][1]
        and BIASI_VALID_RANGE["pressure_Pa"][0] <= pressure_pa
        <= BIASI_VALID_RANGE["pressure_Pa"][1]
        and BIASI_VALID_RANGE["mass_flux_kg_m2s"][0] <= mass_flux_kg_m2s
        <= BIASI_VALID_RANGE["mass_flux_kg_m2s"][1])
    return chf_kW_m2 * 1.0e3, in_range


def coolant_screen(config, cfd_rows):
    if not cfd_rows:
        return {
            "status": "BLOCKED_MISSING_CFD_SWEEP",
            "rows": 0,
            "gates": {
                "hydraulic_screen_pass": None,
                "thermal_capacity_screen_pass": None,
                "wall_temperature_screen_pass": None,
                "coolant_screen_pass": None,
            },
            "reason": "CFD/CHT sweep was not supplied; no coolant result was invented.",
        }, []
    water_verification = verify_water_property_correlations()
    roughness = config["water_roughness_m"]
    length = config["heated_length_m"]
    face_area = config["rf_face_area_m2"]
    routing = config.get("secondary_heat_removal_fraction_by_scenario", {})
    if not isinstance(routing, dict):
        raise ValueError("secondary heat routing must be a scenario-to-fraction mapping")
    pressure_rows = []
    for row in cfd_rows:
        try:
            water = water_properties_TP(
                row["inlet_temperature_K"], row["pressure_Pa"])
        except (TypeError, ValueError) as exc:
            input_blocked(
                "coolant state is outside the verified IAPWS Region-1 screen: "
                f"case={row.get('case_id')}, reason={exc}")
        rho = water["density_kg_m3"]
        mu = water["dynamic_viscosity_Pa_s"]
        k = water["conductivity_W_mK"]
        cp = water["cp_J_kgK"]
        n = row["channel_count"]
        d = row["hydraulic_diameter_m"]
        velocity = row["velocity_m_s"]
        qflux = row["local_heat_flux_W_m2"]
        flow_area = n * d ** 2
        mass_flow = rho * velocity * flow_area
        dp, reynolds, friction = darcy_pressure_drop(
            rho, velocity, d, length, roughness, mu)
        prandtl = cp * mu / k
        nusselt = 0.023 * reynolds ** 0.8 * prandtl ** 0.4
        h = nusselt * k / d
        scenario = row["heat_flux_scenario"]
        secondary_fraction = float(routing.get(scenario, 0.0))
        if not 0.0 <= secondary_fraction <= 1.0:
            raise ValueError(
                "secondary heat removal fraction must be between 0 and 1: "
                f"{scenario}={secondary_fraction}")
        incident_heat_load = qflux * face_area
        secondary_heat_load = incident_heat_load * secondary_fraction
        heat_load = incident_heat_load - secondary_heat_load
        effective_local_heat_flux = heat_load / max(face_area, 1.0e-30)
        bulk_rise = heat_load / max(mass_flow * cp, 1.0)
        wall_rise = effective_local_heat_flux / max(h, 1.0)
        wall_temperature = row["inlet_temperature_K"] + bulk_rise + wall_rise
        declared_dp_error = abs(dp - row["screening_Darcy_pressure_drop_Pa"])
        hydraulic_pass = dp <= row["pressure_drop_limit_Pa"]
        capacity_pass = bulk_rise <= row["allowed_bulk_rise_K"]
        wall_pass = wall_temperature <= row["CuCrZr_temperature_limit_K"]
        bulk_temperature = row["inlet_temperature_K"] + bulk_rise
        mass_flux = mass_flow / max(flow_area, 1.0e-30)
        try:
            quality, tsat_K, hfg_J_kg = local_equilibrium_quality(
                bulk_temperature, row["pressure_Pa"], cp)
            chf_W_m2, chf_in_valid_range = biasi_chf_W_m2(
                row["pressure_Pa"], mass_flux, d, quality)
            dnbr = chf_W_m2 / max(effective_local_heat_flux, 1.0e-30)
            chf_pass = bool(dnbr >= config["minimum_DNBR"])
            chf_status = (
                "BIASI_CORRELATION_SCREEN" if chf_in_valid_range
                else "BIASI_CORRELATION_OUTSIDE_VALIDATED_RANGE")
        except ValueError as exc:
            quality = tsat_K = hfg_J_kg = chf_W_m2 = dnbr = None
            chf_pass = False
            chf_in_valid_range = False
            chf_status = "BLOCKED: %s" % exc
        pressure_rows.append({
            **row,
            "flow_area_m2": flow_area,
            "water_temperature_K": water["temperature_K"],
            "water_pressure_Pa": water["pressure_Pa"],
            "water_density_kg_m3": rho,
            "water_dynamic_viscosity_Pa_s": mu,
            "water_conductivity_W_mK": k,
            "water_cp_J_kgK": cp,
            "water_property_region": water["region"],
            "mass_flow_kg_s": mass_flow,
            "Reynolds": reynolds,
            "Prandtl_screening": prandtl,
            "Darcy_friction_factor": friction,
            "recomputed_pressure_drop_Pa": dp,
            "declared_Darcy_absolute_error_Pa": declared_dp_error,
            "Nusselt_screening": nusselt,
            "h_screening_W_m2K": h,
            "incident_heat_load_W_per_cassette": incident_heat_load,
            "secondary_heat_removal_fraction": secondary_fraction,
            "secondary_heat_routed_W_per_cassette": secondary_heat_load,
            "heat_load_W_per_cassette": heat_load,
            "incident_local_heat_flux_W_m2": qflux,
            "effective_cassette_heat_flux_W_m2": effective_local_heat_flux,
            "heat_balance_residual_W": (
                incident_heat_load - secondary_heat_load - heat_load),
            "bulk_rise_K_screening": bulk_rise,
            "wall_superheat_K_screening": wall_rise,
            "wall_temperature_K_screening": wall_temperature,
            "hydraulic_screen_pass": hydraulic_pass,
            "thermal_capacity_screen_pass": capacity_pass,
            "wall_temperature_screen_pass": wall_pass,
            "screen_pass": bool(
                hydraulic_pass and capacity_pass and wall_pass and chf_pass),
            "value_status": "PROVISIONAL_CORRELATION_SCREEN_NOT_CFD_OR_CHF_RESULT",
            "mass_flux_kg_m2s": mass_flux,
            "local_equilibrium_quality_screening": quality,
            "saturation_temperature_K_IAPWS_IF97": tsat_K,
            "latent_heat_J_kg_Watson_estimate": hfg_J_kg,
            "biasi_chf_W_m2": chf_W_m2,
            "DNBR_screening": dnbr,
            "biasi_valid_range_pass": chf_in_valid_range,
            "chf_screen_pass": chf_pass,
            "chf_status": chf_status,
        })
    hydraulic = all(row["hydraulic_screen_pass"] for row in pressure_rows)
    capacity = all(row["thermal_capacity_screen_pass"] for row in pressure_rows)
    wall = all(row["wall_temperature_screen_pass"] for row in pressure_rows)
    chf_gradeable = [row for row in pressure_rows if row["DNBR_screening"] is not None]
    chf_pass_all = bool(
        chf_gradeable and all(row["chf_screen_pass"] for row in chf_gradeable)
        and len(chf_gradeable) == len(pressure_rows))
    chf_all_in_valid_range = bool(
        chf_gradeable and all(row["biasi_valid_range_pass"] for row in chf_gradeable))
    scenarios = {}
    for scenario in sorted({row["heat_flux_scenario"] for row in pressure_rows}):
        subset = [row for row in pressure_rows if row["heat_flux_scenario"] == scenario]
        subset_chf = [row for row in subset if row["DNBR_screening"] is not None]
        scenarios[scenario] = {
            "rows": len(subset),
            "secondary_heat_removal_fraction": sorted({
                row["secondary_heat_removal_fraction"] for row in subset}),
            "max_incident_heat_load_W_per_cassette": max(
                row["incident_heat_load_W_per_cassette"] for row in subset),
            "max_secondary_heat_routed_W_per_cassette": max(
                row["secondary_heat_routed_W_per_cassette"] for row in subset),
            "max_local_heat_load_W_per_cassette": max(
                row["heat_load_W_per_cassette"] for row in subset),
            "max_heat_balance_residual_W": max(abs(
                row["heat_balance_residual_W"]) for row in subset),
            "max_pressure_drop_Pa": max(row["recomputed_pressure_drop_Pa"] for row in subset),
            "max_bulk_rise_K": max(row["bulk_rise_K_screening"] for row in subset),
            "max_wall_temperature_K": max(
                row["wall_temperature_K_screening"] for row in subset),
            "min_DNBR_screening": (
                min(row["DNBR_screening"] for row in subset_chf)
                if subset_chf else None),
            "screen_pass": bool(all(row["screen_pass"] for row in subset)),
        }
    summary = {
        "rows": len(pressure_rows),
        "max_pressure_drop_Pa": max(
            row["recomputed_pressure_drop_Pa"] for row in pressure_rows),
        "max_bulk_rise_K": max(row["bulk_rise_K_screening"] for row in pressure_rows),
        "max_wall_temperature_K": max(
            row["wall_temperature_K_screening"] for row in pressure_rows),
        "maximum_declared_Darcy_check_error_Pa": max(
            row["declared_Darcy_absolute_error_Pa"] for row in pressure_rows),
        "min_DNBR_screening": (
            min(row["DNBR_screening"] for row in chf_gradeable)
            if chf_gradeable else None),
        "minimum_DNBR_required": config["minimum_DNBR"],
        "secondary_heat_routing": {
            "fractions_by_scenario": {
                str(key): float(value) for key, value in routing.items()},
            "provenance": config.get("secondary_heat_removal_provenance"),
            "energy_balance": (
                "incident heat = secondary routed heat + cassette-local heat"),
            "secondary_system_capacity_status": (
                "NOT_MODELED; routed heat is reported but not qualified"),
        },
        "gates": {
            "hydraulic_screen_pass": hydraulic,
            "thermal_capacity_screen_pass": capacity,
            "wall_temperature_screen_pass": wall,
            "chf_correlation_screen_pass": chf_pass_all,
            "chf_correlation_within_validated_range": chf_all_in_valid_range,
            "water_property_model_pass": True,
            "coolant_screen_pass": bool(
                hydraulic and capacity and wall and chf_pass_all),
        },
        "scenario_summary": scenarios,
        "water_properties_status": water_verification["status"],
        "water_property_model": {
            "standards": {
                "thermodynamics": "IAPWS-IF97 Region 1",
                "viscosity": "IAPWS 2008 industrial branch (mu2=1)",
                "thermal_conductivity": "IAPWS 2011 industrial branch (lambda2=0)",
            },
            "verification": water_verification,
            "state_policy": (
                "T/P states outside IF97 Region 1 are blocked; no constant-value "
                "fallback is permitted."),
        },
        "CHF_status": (
            "BIASI_1967_CORRELATION_SCREEN_NOT_A_QUALIFIED_CHF_RESULT" if chf_all_in_valid_range
            else "BIASI_1967_CORRELATION_SCREEN_SOME_ROWS_OUTSIDE_VALIDATED_RANGE"),
        "CHF_provenance": (
            "Biasi (1967) empirical CHF correlation; local equilibrium quality "
            "estimated from bulk subcooling via a Watson-correlation latent-heat "
            "estimate, not a full IAPWS steam-table enthalpy balance. This "
            "narrows but does not remove the qualified_CHF external-solver "
            "blocker -- it is a validated *correlation*, not a certified "
            "CHF test or a mechanistic sub-channel code result."),
        "status": "IAPWS_PROPERTY_COOLANT_HYDRAULIC_AND_ENERGY_SCREEN",
    }
    return summary, pressure_rows


def uncertainty_sweep(config, equilibrium, confinement, gates, output,
                      cases=65536, seed=20260905, chunk_size=4096):
    """Run a memory-bounded reduced envelope screen; no new GS solve per sample."""
    output = Path(output)
    output.mkdir(parents=True, exist_ok=True)
    fields = [
        "case_id", "density_scale", "Ti_scale", "Te_scale", "B0_scale",
        "tau_E_scale", "alpha_deposition_fraction", "Z_eff", "a_scale",
        "R0_scale", "Ip_scale", "fusion_GW_module", "alpha_ratio",
        "tau_E_required_s", "tau_E_margin_ratio", "beta_axis_percent",
        "beta_N_proxy", "q95_proxy", "gate_all",
    ]
    path = output / "reduced_uncertainty_cases.csv"
    rng = np.random.default_rng(int(seed))
    base_fusion = confinement["fusion_GW_module"] * 1.0e9
    base_brem = confinement["bremsstrahlung_GW_module"] * 1.0e9
    base_w = confinement["thermal_energy_GJ_module"] * 1.0e9
    ti0 = confinement["volume_weighted_Ti_keV"]
    base_reactivity = float(dt_reactivity(ti0))
    p_avg = confinement["volume_weighted_density_m3"] * (
        confinement["volume_weighted_Ti_keV"]
        + confinement["volume_weighted_Te_keV"]) * KEV_TO_J
    base_beta_axis = 100.0 * 2.0 * MU0 * p_avg / config["B0_plasma_T"] ** 2
    base_beta_n = equilibrium["beta_N_proxy"]
    base_q95 = equilibrium["q95_cylindrical_proxy"]
    base_eq_pass = bool(
        equilibrium["gates"]["equilibrium_numerical_pass"]
        and equilibrium["gates"]["pressure_monotonic_pass"])
    reference_reconciliation_pass = gates.get(
        "reference_self_heating_reconciliation_pass")
    reference_reconciliation_ok = reference_reconciliation_pass is True
    static_screen_ok = bool(
        gates.get("tf_ripple_screen_pass", True) is True
        and gates.get("water_property_model_pass", True) is True)
    passes = 0
    best_ratio = -math.inf
    best_margin = -math.inf
    best_q95 = -math.inf
    plot_x, plot_y = [], []
    with path.open("w", newline="", encoding="utf-8") as handle:
        writer = csv.DictWriter(handle, fieldnames=fields)
        writer.writeheader()
        for start in range(0, int(cases), int(chunk_size)):
            n = min(int(chunk_size), int(cases) - start)
            ds = rng.uniform(0.80, 1.30, n)
            tis = rng.uniform(0.80, 1.30, n)
            tes = rng.uniform(0.80, 1.30, n)
            bs = rng.uniform(0.85, 1.15, n)
            taus = rng.uniform(0.60, 3.00, n)
            alpha_dep = rng.uniform(0.80, 0.98, n)
            zeff = rng.uniform(1.00, 1.80, n)
            a_scale = rng.uniform(0.87, 1.13, n)
            r_scale = rng.uniform(0.91, 1.09, n)
            ip_scale = rng.uniform(0.72, 1.28, n)
            fusion = base_fusion * ds ** 2 * (
                dt_reactivity(ti0 * tis) / max(base_reactivity, 1.0e-99))
            brem = base_brem * ds ** 2 * np.sqrt(tes) * zeff
            thermal = base_w * ds * 0.5 * (tis + tes)
            transport = thermal / taus
            alpha_power = fusion * ALPHA_FRACTION * alpha_dep
            ratio = alpha_power / np.maximum(brem + transport, 1.0)
            tau_req = np.where(alpha_power > brem,
                               thermal / np.maximum(alpha_power - brem, 1.0),
                               np.inf)
            margin = np.where(np.isfinite(tau_req), taus / tau_req - 1.0,
                              -np.inf)
            beta_axis = base_beta_axis * ds * 0.5 * (tis + tes) / bs ** 2
            beta_n = base_beta_n * ds * 0.5 * (tis + tes) * a_scale / (
                bs * ip_scale)
            q95 = base_q95 * a_scale ** 2 * bs / (r_scale * ip_scale)
            gate_all = (
                base_eq_pass
                & static_screen_ok
                & reference_reconciliation_ok
                & (q95 >= config["q95_proxy_min"])
                & (beta_n <= config["beta_N_proxy_max"])
                & (beta_axis <= config["beta_axis_proxy_max_percent"])
                & (ratio >= 1.0)
                & (margin >= config["tau_E_margin_min"])
            )
            passes += int(np.sum(gate_all))
            best_ratio = max(best_ratio, float(np.max(ratio)))
            finite_margin = margin[np.isfinite(margin)]
            if len(finite_margin):
                best_margin = max(best_margin, float(np.max(finite_margin)))
            best_q95 = max(best_q95, float(np.max(q95)))
            if len(plot_x) < 12000:
                take = min(n, 12000 - len(plot_x))
                plot_x.extend(q95[:take].tolist())
                plot_y.extend(beta_n[:take].tolist())
            for index in range(n):
                writer.writerow({
                    "case_id": start + index,
                    "density_scale": ds[index],
                    "Ti_scale": tis[index],
                    "Te_scale": tes[index],
                    "B0_scale": bs[index],
                    "tau_E_scale": taus[index],
                    "alpha_deposition_fraction": alpha_dep[index],
                    "Z_eff": zeff[index],
                    "a_scale": a_scale[index],
                    "R0_scale": r_scale[index],
                    "Ip_scale": ip_scale[index],
                    "fusion_GW_module": fusion[index] / 1.0e9,
                    "alpha_ratio": ratio[index],
                    "tau_E_required_s": tau_req[index],
                    "tau_E_margin_ratio": margin[index],
                    "beta_axis_percent": beta_axis[index],
                    "beta_N_proxy": beta_n[index],
                    "q95_proxy": q95[index],
                    "gate_all": bool(gate_all[index]),
                })
    summary = {
        "cases": int(cases),
        "seed": int(seed),
        "all_gate_passes": passes,
        "all_gate_pass_percent": 100.0 * passes / max(int(cases), 1),
        "best_alpha_self_heating_ratio": best_ratio,
        "best_tau_E_margin_ratio": best_margin,
        "best_q95_proxy": best_q95,
        "note": "Reduced scaling screen; it does not re-solve Grad-Shafranov per sample.",
        "status": "REDUCED_UNCERTAINTY_SCREEN",
    }
    write_json(output / "reduced_uncertainty_summary.json", summary)
    return summary, np.asarray(plot_x), np.asarray(plot_y)


def write_csv(path, rows):
    rows = list(rows)
    if not rows:
        return
    with Path(path).open("w", newline="", encoding="utf-8") as handle:
        writer = csv.DictWriter(handle, fieldnames=list(rows[0].keys()))
        writer.writeheader()
        writer.writerows(json_safe(row) for row in rows)


def make_plots(output, grid, fields, uncertainty_x, uncertainty_y):
    try:
        import matplotlib
        matplotlib.use("Agg")
        import matplotlib.pyplot as plt
    except ImportError:
        return {"plots_status": "MATPLOTLIB_UNAVAILABLE"}
    output = Path(output)
    RR, ZZ, mask = grid["RR"], grid["ZZ"], grid["mask"]
    fig, axes = plt.subplots(1, 3, figsize=(15, 4.6), constrained_layout=True)
    psi = np.where(mask, fields["psi"], np.nan)
    pressure = np.where(mask, fields["pressure"] / 1e6, np.nan)
    current = np.where(mask, fields["JPHI"] / 1e6, np.nan)
    levels = np.linspace(float(np.nanmin(psi)), float(np.nanmax(psi)), 18)
    contours = axes[0].contour(RR, ZZ, psi, levels=levels, cmap="viridis")
    axes[0].clabel(contours, inline=True, fontsize=7, fmt="%.2g")
    axes[0].set_title("Fixed-boundary poloidal flux")
    image = axes[1].pcolormesh(RR, ZZ, pressure, shading="auto", cmap="magma")
    fig.colorbar(image, ax=axes[1], label="MPa")
    axes[1].set_title("Pressure screen")
    image = axes[2].pcolormesh(RR, ZZ, current, shading="auto", cmap="coolwarm")
    fig.colorbar(image, ax=axes[2], label="MA/m2")
    axes[2].set_title("Toroidal current-density screen")
    for axis in axes:
        axis.set_xlabel("R (m)")
        axis.set_ylabel("Z (m)")
    fig.savefig(output / "equilibrium_fields.png", dpi=160)
    plt.close(fig)

    if len(uncertainty_x):
        fig, axis = plt.subplots(figsize=(6.6, 5.0), constrained_layout=True)
        axis.scatter(uncertainty_x, uncertainty_y, s=3, alpha=0.22)
        axis.axvline(DEFAULTS["q95_proxy_min"], color="black", linestyle="--")
        axis.axhline(DEFAULTS["beta_N_proxy_max"], color="black", linestyle="--")
        axis.set_xlabel("q95 cylindrical proxy")
        axis.set_ylabel("beta_N proxy")
        axis.set_title("Reduced uncertainty MHD-proxy space")
        fig.savefig(output / "reduced_uncertainty_mhd_space.png", dpi=160)
        plt.close(fig)
    return {"plots_status": "GENERATED"}




# ---------------------------------------------------------------------------
# Local ideal-MHD interchange/ballooning stability proxy (Suydam + s-alpha).
#
# This narrows -- but does NOT remove -- the "nonlinear_MHD_stability_spectrum"
# and "free_boundary_3D_MHD" external-solver blockers below. It is a local,
# large-aspect-ratio, circular-cross-section analytic screen, not a full
# n -> infinity ballooning eigenvalue solve. See the docstrings for sources
# and explicit limitations.
# ---------------------------------------------------------------------------

# Large-aspect-ratio, circular-cross-section first-stability ballooning
# threshold: alpha_crit(s) ~= BALLOONING_ALPHA_CRIT_COEFFICIENT * s.
# Source: Connor, Hastie & Taylor, PRL 40, 396 (1978); see also Freidberg,
# "Ideal Magnetohydrodynamics" (2014) sec. 10.6, and Wesson, "Tokamaks"
# sec. 7.9-7.10 for the s-alpha diagram. The coefficient is O(1) and shape
# dependent in real (non-circular, finite-aspect-ratio) equilibria -- treat
# this as a screening threshold, not a validated boundary.
BALLOONING_ALPHA_CRIT_COEFFICIENT = 0.6

# Surfaces with |shear| below this are flagged instead of graded, since the
# linear s-alpha threshold is not meaningful near s = 0 and the sign
# convention for negative shear needs a dedicated (non-local) treatment.
MIN_GRADEABLE_SHEAR = 0.05

# Innermost/outermost normalized-flux fraction excluded from the gate
# because area-derivative q(psi) is numerically noisy right at the axis
# (small-area cells) and right at the fixed boundary (Dirichlet edge).
EDGE_EXCLUSION_PSI_N = 0.04


def flux_surface_q_profile(grid, fields, n_thresholds=64):
    """Compute q(psi), shear, and an effective minor-radius label.

    Uses q(psi) = d(Phi_toroidal)/d(psi), which is exact for an
    axisymmetric equilibrium and needs only area integrals on the existing
    (R, Z) grid -- no contour tracing, no new dependency.

    Returns a dict of 1-D arrays, ordered from axis (index 0) to boundary
    (index -1):
        psi_n            normalized poloidal flux, 1 at axis, 0 at boundary
        psi              psi in Wb at each surface
        r_eff_m          effective minor radius sqrt(Area/pi), meters
        rho_eff          r_eff_m / a_m (for comparison with the existing
                          geometric "rho" label used elsewhere in Tomahawk)
        q                safety factor magnitude |dPhi_tor/dpsi|
        bphi_avg_T       flux-surface-averaged toroidal field, tesla
        pressure_avg_Pa  flux-surface-averaged pressure, pascal
    """
    mask = grid["mask"]
    dR, dZ = grid["dR"], grid["dZ"]
    dA = dR * dZ
    psi = fields["psi"]
    bphi = fields["BPHI"]
    pressure = fields["pressure"]

    psi_axis = float(np.max(psi[mask]))
    if not math.isfinite(psi_axis) or psi_axis <= 0.0:
        raise RuntimeError(
            "flux_surface_q_profile requires a positive finite psi_axis")

    psi_n_grid = np.clip(psi / psi_axis, 0.0, 1.0)
    flat_psin = psi_n_grid[mask]
    flat_bphi = bphi[mask]
    flat_pressure = pressure[mask]

    # Thresholds from near-boundary to near-axis; each threshold t defines
    # the surface psi_n = t, enclosing the region psi_n >= t.
    thresholds = np.linspace(1.0 - EDGE_EXCLUSION_PSI_N,
                             EDGE_EXCLUSION_PSI_N, int(n_thresholds))
    area = np.empty_like(thresholds)
    phi_tor = np.empty_like(thresholds)
    bphi_avg = np.empty_like(thresholds)
    pressure_avg = np.empty_like(thresholds)
    shell_half_width = 0.5 * (thresholds[0] - thresholds[1]) if len(thresholds) > 1 else 0.01

    for i, t in enumerate(thresholds):
        enclosed = flat_psin >= t
        n_enclosed = int(np.count_nonzero(enclosed))
        if n_enclosed < 4:
            area[i] = np.nan
            phi_tor[i] = np.nan
        else:
            area[i] = n_enclosed * dA
            phi_tor[i] = float(np.sum(flat_bphi[enclosed])) * dA
        shell = np.abs(flat_psin - t) <= shell_half_width
        if np.count_nonzero(shell) < 2:
            shell = enclosed & (flat_psin <= t + 2.0 * shell_half_width)
        bphi_avg[i] = float(np.mean(flat_bphi[shell])) if np.any(shell) else np.nan
        pressure_avg[i] = (
            float(np.mean(flat_pressure[shell])) if np.any(shell) else np.nan)

    psi_at_threshold = thresholds * psi_axis
    # dPhi_tor/dpsi via central differences on the (now monotonic-in-psi)
    # sampled arrays; thresholds run axis->boundary is not guaranteed
    # monotonic in psi if psi is not perfectly monotonic on the grid, so
    # sort explicitly by psi before differentiating.
    order = np.argsort(psi_at_threshold)
    psi_sorted = psi_at_threshold[order]
    phi_sorted = phi_tor[order]
    valid = np.isfinite(psi_sorted) & np.isfinite(phi_sorted)
    q_sorted = np.full_like(psi_sorted, np.nan)
    if np.count_nonzero(valid) >= 3:
        # NOTE: this code's psi is the flux-per-radian convention (i.e.
        # Br = -(1/R) d(psi)/dZ, Bz = (1/R) d(psi)/dR with no explicit 2*pi
        # anywhere), matching Wesson/Freidberg's normalized poloidal flux
        # psi = Psi_actual/(2*pi). The safety factor is defined against the
        # *actual* poloidal flux, q = dPhi_tor/dPsi_actual
        # = (1/(2*pi)) * dPhi_tor/dpsi -- the 1/(2*pi) is required here and
        # is not optional bookkeeping.
        q_sorted[valid] = np.abs(
            np.gradient(phi_sorted[valid], psi_sorted[valid], edge_order=2)
        ) / (2.0 * math.pi)
    # Undo the sort to return arrays ordered axis -> boundary (thresholds
    # order), matching area/bphi_avg/pressure_avg.
    unsort = np.argsort(order)
    q = q_sorted[unsort]
    r_eff_m = np.sqrt(np.maximum(area, 0.0) / math.pi)

    return {
        "psi_n": thresholds,
        "psi_Wb": psi_at_threshold,
        "r_eff_m": r_eff_m,
        "q": q,
        "bphi_avg_T": bphi_avg,
        "pressure_avg_Pa": pressure_avg,
    }


def mercier_ballooning_proxy(config, grid, fields, n_thresholds=64):
    """Evaluate the Suydam interchange and s-alpha ballooning proxies.

    Returns (metrics_dict, gates_dict). See module docstring for the exact
    criteria, their sources, and their explicit limitations.
    """
    profile = flux_surface_q_profile(grid, fields, n_thresholds=n_thresholds)
    r = profile["r_eff_m"]
    q = profile["q"]
    bphi = profile["bphi_avg_T"]
    p = profile["pressure_avg_Pa"]
    a_m = float(config["a_m"])
    r0_m = float(config["R0_m"])
    b0_t = float(config["B0_plasma_T"])

    valid = (np.isfinite(r) & np.isfinite(q) & np.isfinite(bphi)
             & np.isfinite(p) & (r > 1.0e-6) & (q > 0.0))
    order = np.argsort(r)
    r_s, q_s, bphi_s, p_s, valid_s = (
        r[order], q[order], bphi[order], p[order], valid[order])

    n_pts = int(np.count_nonzero(valid_s))
    if n_pts < 5:
        return (
            {
                "status": "INSUFFICIENT_VALID_SURFACES",
                "n_valid_surfaces": n_pts,
            },
            {
                "suydam_interchange_proxy_pass": False,
                "ballooning_s_alpha_proxy_pass": False,
            },
        )

    r_v = r_s[valid_s]
    q_v = q_s[valid_s]
    bphi_v = bphi_s[valid_s]
    p_v = p_s[valid_s]

    dq_dr = np.gradient(q_v, r_v, edge_order=2)
    dp_dr = np.gradient(p_v, r_v, edge_order=2)
    shear = (r_v / q_v) * dq_dr

    # Suydam (large-aspect-ratio Mercier limit), SI units:
    #   D_Suydam = (r * Bphi^2 / 4) * (dlnq/dr)^2 + mu0 * dp/dr  >= 0
    dlnq_dr = dq_dr / q_v
    suydam = (r_v * bphi_v ** 2 / 4.0) * dlnq_dr ** 2 + MU0 * dp_dr

    # s-alpha ballooning parameter: alpha = -(2 mu0 R0 q^2 / B0^2) dp/dr
    alpha = -(2.0 * MU0 * r0_m * q_v ** 2 / b0_t ** 2) * dp_dr
    alpha_crit = BALLOONING_ALPHA_CRIT_COEFFICIENT * np.abs(shear)

    gradeable = np.abs(shear) >= MIN_GRADEABLE_SHEAR
    n_gradeable = int(np.count_nonzero(gradeable))
    n_ungradeable = n_pts - n_gradeable

    suydam_pass = bool(np.all(suydam >= 0.0)) if n_pts else False
    if n_gradeable > 0:
        ballooning_pass = bool(np.all(alpha[gradeable] <= alpha_crit[gradeable]))
        ballooning_margin = float(np.min(
            (alpha_crit[gradeable] - alpha[gradeable])
            / np.maximum(alpha_crit[gradeable], 1.0e-12)))
    else:
        ballooning_pass = False
        ballooning_margin = float("nan")

    worst_idx = int(np.argmin(suydam)) if n_pts else -1
    metrics = {
        "n_surfaces_evaluated": n_pts,
        "n_low_shear_surfaces_unassessed": n_ungradeable,
        "rho_eff_range": [float(np.min(r_v) / a_m), float(np.max(r_v) / a_m)],
        "min_suydam_D": float(np.min(suydam)),
        "min_suydam_D_at_rho_eff": float(r_v[worst_idx] / a_m),
        "max_ballooning_alpha_over_alpha_crit": (
            float(np.max(alpha[gradeable] / np.maximum(alpha_crit[gradeable], 1.0e-12)))
            if n_gradeable else None),
        "ballooning_stability_margin_min": (
            ballooning_margin if n_gradeable else None),
        "ballooning_alpha_crit_coefficient": BALLOONING_ALPHA_CRIT_COEFFICIENT,
        "ballooning_alpha_crit_provenance": (
            "large-aspect-ratio circular-plasma s-alpha first-stability "
            "threshold, Connor-Hastie-Taylor PRL 40, 396 (1978); screening "
            "threshold only, not shape-corrected"),
        "min_gradeable_shear": MIN_GRADEABLE_SHEAR,
        "method": (
            "q(psi) = d(Phi_toroidal)/d(psi) via area integration on the "
            "existing axisymmetric grid; Suydam and s-alpha criteria "
            "evaluated at each resolved flux surface"),
        "limitations": [
            "Large-aspect-ratio, circular-cross-section analytic criteria; "
            "no shape/elongation/triangularity correction to the "
            "ballooning threshold.",
            "Not a full n -> infinity ballooning eigenvalue solve; no "
            "second-stability access assessment.",
            "Low-shear and negative-shear surfaces are excluded from the "
            "pass/fail grade, not certified stable.",
            "Ideal MHD only: no resistive, kinetic, or non-axisymmetric "
            "(n != 0) mode assessment.",
        ],
        "status": "LOCAL_INTERCHANGE_AND_BALLOONING_SCREENING_PROXY",
    }
    gates = {
        "suydam_interchange_proxy_pass": suydam_pass,
        "ballooning_s_alpha_proxy_pass": ballooning_pass,
    }
    return metrics, gates


# ---------------------------------------------------------------------------
# Discrete-coil (16x, filamentary) TF ripple screen.
#
# IMPORTANT SCOPE NOTE: this is a toroidal-field-ripple diagnostic, not a
# free-boundary POLOIDAL equilibrium solve. TF coils set/ripple the vacuum
# toroidal field; they do not shape the poloidal flux surface (that needs
# PF/shaping coils, which are not present in this dataset). The CAD-A2 branch
# is a controlling screening gate, while the legacy branch remains advisory.
#
# CRITICAL PROVENANCE NOTE, per MHD/cad_kaggle_reconciliation.json:
#   - The Kaggle output supplies a 100-turn reference count and legacy
#     321547.59 A schedule, but that schedule was sized on the legacy
#     centerline and cannot be reused unchanged.
#   - This release contains a separate CAD-A2 centerline and a new
#     model-derived current schedule obtained by rerunning the same
#     Biot-Savart B0 target on that CAD-A2 centerline.
#   - The legacy branch is retained only as a comparison artifact. It is
#     never mixed with the CAD-A2 branch or used as its gate.
#
# Field method: exact closed-form Biot-Savart law for straight filamentary
# segments (Hanson & Hirshman, Phys. Plasmas 9, 4410 (2002)), summed over
# all coil segments. Validated against the exact analytic on-axis field of
# a circular current loop (agreement to 7 significant figures).
# ---------------------------------------------------------------------------
def biot_savart_segments(p1, p2, current, field_points):
    """Exact field of a straight filamentary segment (p1->p2, current I) at
    each row of field_points (N,3). Hanson & Hirshman (2002) closed form."""
    a = p1[None, :] - field_points
    b = p2[None, :] - field_points
    amag = np.linalg.norm(a, axis=1)
    bmag = np.linalg.norm(b, axis=1)
    cross_ab = np.cross(a, b)
    denom = amag * bmag * (amag * bmag + np.sum(a * b, axis=1))
    denom = np.where(denom == 0, np.inf, denom)
    factor = (amag + bmag) / denom
    return (MU0 * current / (4.0 * np.pi)) * factor[:, None] * cross_ab


def load_tf_coil_bundle(centerlines_path, currents_path, geometry_branch,
                        current_derivation):
    """Load a filament branch and its matching flat-top current schedule."""
    import csv as _csv
    segments_by_coil = {}
    with open(centerlines_path, newline="") as fh:
        for row in _csv.DictReader(fh):
            cid = row["coil_id"]
            segments_by_coil.setdefault(cid, []).append((
                float(row["x_m"]), float(row["y_m"]), float(row["z_m"]),
                float(row["next_x_m"]), float(row["next_y_m"]), float(row["next_z_m"]),
                int(row["turns"])))
    currents_rows = []
    with open(currents_path, newline="") as fh:
        for row in _csv.DictReader(fh):
            currents_rows.append((
                float(row["time_s"]), row["coil_id"], float(row["current_A"])))
    if not segments_by_coil or not currents_rows:
        input_blocked(
            f"{geometry_branch} TF coil centerline or current file was empty")
    flattop_time = max(r[0] for r in currents_rows)
    flattop_current = {
        cid: amps for (t, cid, amps) in currents_rows if t == flattop_time}
    if set(flattop_current) != set(segments_by_coil):
        input_blocked(
            f"{geometry_branch} TF coil set mismatch between centerlines and currents: "
            f"{sorted(set(segments_by_coil))} vs {sorted(set(flattop_current))}")
    p1_list, p2_list, i_list = [], [], []
    radii = []
    for cid, segs in segments_by_coil.items():
        turns = segs[0][6]
        if any(s[6] != turns for s in segs):
            input_blocked(f"inconsistent turns count within coil {cid}")
        i_eff = flattop_current[cid] * turns
        for (x, y, z, nx, ny, nz, _turns) in segs:
            p1_list.append((x, y, z))
            p2_list.append((nx, ny, nz))
            i_list.append(i_eff)
            radii.append(math.hypot(x, y))
            radii.append(math.hypot(nx, ny))
    return {
        "P1": np.array(p1_list), "P2": np.array(p2_list), "I": np.array(i_list),
        "coil_count": len(segments_by_coil),
        "flattop_time_s": flattop_time,
        "flattop_current_A": sorted(set(flattop_current.values())),
        "turns_per_coil": sorted({segs[0][6] for segs in segments_by_coil.values()}),
        "coil_inner_leg_R_m": min(radii),
        "coil_outer_leg_R_m": max(radii),
        "geometry_branch": geometry_branch,
        "current_derivation": current_derivation,
    }


def load_legacy_tf_coil_bundle(centerlines_path, currents_path):
    """Load the recovered legacy branch for comparison only."""
    return load_tf_coil_bundle(
        centerlines_path, currents_path,
        "LEGACY_KAGGLE_MODEL_DERIVED_FILAMENT_CENTERLINE",
        "KAGGLE_OUTPUT_LEGACY_B0_TARGET")


def load_cad_a2_tf_coil_bundle(centerlines_path, currents_path):
    """Load the CAD-A2 geometry with its explicit model-derived rerun schedule."""
    return load_tf_coil_bundle(
        centerlines_path, currents_path,
        "CAD_A2_MODEL_DERIVED_CENTERLINE",
        "KAGGLE_100_TURN_REFERENCE_REMAP_AND_CAD_A2_BIOT_SAVART_B0_RERUN")


def tf_coil_field_at(coil_bundle, points_xyz):
    """Real B-field (Nx3, Tesla) at points_xyz (Nx3) from the legacy TF
    coil filament set. O(n_segments) per call; fine for a handful of
    ripple-scan points, not intended for full 2D grid evaluation."""
    P1, P2, I = coil_bundle["P1"], coil_bundle["P2"], coil_bundle["I"]
    field = np.zeros_like(points_xyz, dtype=float)
    for i in range(len(P1)):
        field += biot_savart_segments(P1[i], P2[i], I[i], points_xyz)
    return field


def tf_ripple_screen(config, coil_bundle, n_phi=64):
    """Evaluate one internally consistent TF geometry/current branch."""
    r0 = float(config["R0_m"])
    a = float(config["a_m"])
    coil_count = coil_bundle["coil_count"]
    period_deg = 360.0 / coil_count

    def ripple_at_radius(radius_m):
        phis = np.linspace(0.0, np.radians(period_deg), n_phi)
        pts = np.stack(
            [radius_m * np.cos(phis), radius_m * np.sin(phis), np.zeros_like(phis)],
            axis=1)
        b = tf_coil_field_at(coil_bundle, pts)
        bphi = np.abs(-b[:, 0] * np.sin(phis) + b[:, 1] * np.cos(phis))
        ripple_pct = 100.0 * (bphi.max() - bphi.min()) / (bphi.max() + bphi.min())
        return {
            "radius_m": radius_m,
            "Bphi_min_T": float(bphi.min()),
            "Bphi_max_T": float(bphi.max()),
            "Bphi_mean_T": float(bphi.mean()),
            "ripple_percent": float(ripple_pct),
        }

    inner_leg_R = coil_bundle["coil_inner_leg_R_m"]
    outer_leg_R = coil_bundle["coil_outer_leg_R_m"]
    plasma_inboard_R = r0 - a
    plasma_outboard_R = r0 + a
    inboard_clearance_m = plasma_inboard_R - inner_leg_R
    outboard_clearance_m = outer_leg_R - plasma_outboard_R

    axis_result = ripple_at_radius(r0)
    edge_result = None
    edge_status = "NOT_EVALUATED"
    if inboard_clearance_m > 0.0 and outboard_clearance_m > 0.0:
        edge_result = ripple_at_radius(plasma_outboard_R)
        edge_status = "EVALUATED"
    else:
        edge_status = "SKIPPED_NEGATIVE_CLEARANCE_COIL_PLASMA_OVERLAP"

    threshold = float(config["tf_ripple_screening_max_percent"])
    ripple_pass_at_edge = (
        bool(edge_result["ripple_percent"] <= threshold) if edge_result else False)

    branch = coil_bundle.get("geometry_branch", "UNDECLARED_BRANCH")
    is_cad_a2 = branch == "CAD_A2_MODEL_DERIVED_CENTERLINE"
    return {
        "geometry_branch": branch,
        "current_derivation": coil_bundle.get("current_derivation"),
        "scope_note": (
            "TF (toroidal-field) ripple diagnostic only. Does NOT constitute "
            "a free-boundary POLOIDAL equilibrium solve; no PF/shaping coil "
            "data was supplied. PF/free-boundary closure remains a separate "
            "blocked gate."
        ),
        "provenance_warning": (
            "CAD-A2 uses the exact CAD-A2 centerline generated from the Rev1.7 "
            "winding envelope and a separate model-derived current rerun using "
            "the 100-turn reference count recovered from the Kaggle output. "
            "It is not an as-built electrical schedule."
            if is_cad_a2 else
            "Legacy branch retained only as a non-controlling comparison; its "
            "current and centerline are not combined with CAD-A2 geometry."
        ),
        "method": (
            "Exact closed-form Biot-Savart (Hanson & Hirshman 2002) summed "
            "over all real coil filament segments; validated against the "
            "analytic on-axis field of a circular loop to 7 significant "
            "figures (see accompanying test)."),
        "coil_count": coil_count,
        "turns_per_coil": coil_bundle["turns_per_coil"],
        "flattop_current_A": coil_bundle["flattop_current_A"],
        "coil_inner_leg_R_m": inner_leg_R,
        "coil_outer_leg_R_m": outer_leg_R,
        "plasma_inboard_R_m": plasma_inboard_R,
        "plasma_outboard_R_m": plasma_outboard_R,
        "inboard_clearance_m": inboard_clearance_m,
        "outboard_clearance_m": outboard_clearance_m,
        "ripple_at_axis": axis_result,
        "ripple_at_plasma_outboard_edge": edge_result,
        "edge_evaluation_status": edge_status,
        "ripple_screening_threshold_percent": threshold,
        "ripple_screening_threshold_provenance": (
            "Literature range ~0.3-0.5% (ITER; de Vries et al., Nucl. Fusion "
            "48, 035007, 2008) up to ~2% tolerated in tight-aspect-ratio "
            "compact devices before mitigation (FAST, ORNL-EPR). This is a "
            "provisional screening threshold, not a certified design limit."),
        "screening_gates": {
            "tf_coil_plasma_clearance_positive": bool(
                inboard_clearance_m > 0.0 and outboard_clearance_m > 0.0),
            "tf_ripple_within_screening_threshold_at_plasma_edge": ripple_pass_at_edge,
        },
        "controlling_gate": is_cad_a2,
        "status": (
            "CAD_A2_MODEL_DERIVED_SCREENING_RESULT"
            if is_cad_a2 else "LEGACY_BRANCH_ADVISORY_ONLY_NOT_A_CONTROLLING_GATE"),
    }


def run_native_screen(input_root=None, output_root="tomahawk_rev24_screen",
                      label="full", cases=65536, seed=20260905, nr=161, nz=161,
                      max_gs_iters=12000, overrides=None):
    """Run one labeled screen and write all auditable artifacts."""
    started = time.time()
    output = Path(output_root).resolve()
    output.mkdir(parents=True, exist_ok=True)
    bundle = load_bundle(input_root)
    config = dict(DEFAULTS)
    mhd_parameters = bundle["mhd_input"].get("screening_parameters", {})
    if mhd_parameters.get("magnetic_field_target_T") is not None:
        config["Bpeak_coil_T"] = float(mhd_parameters["magnetic_field_target_T"])
    active_tau_target = mhd_parameters.get("tau_E_screening_target_s")
    if active_tau_target is not None:
        config["tau_E_s"] = float(active_tau_target)
        config["active_tau_E_screening_target_provenance"] = str(
            mhd_parameters.get(
                "tau_E_screening_target_provenance",
                "active tau-E screening target supplied by MHD input"))
    else:
        config["active_tau_E_screening_target_provenance"] = (
            "default active tau-E screening target; not a measured confinement result")
    secondary_fractions = mhd_parameters.get(
        "secondary_heat_removal_fraction_by_scenario")
    if isinstance(secondary_fractions, dict):
        config["secondary_heat_removal_fraction_by_scenario"] = {
            str(key): float(value)
            for key, value in secondary_fractions.items()}
        config["secondary_heat_removal_provenance"] = str(
            mhd_parameters.get(
                "secondary_heat_removal_provenance",
                config["secondary_heat_removal_provenance"]))
    reference_correction = mhd_parameters.get(
        "reference_condition_correction", {})
    if isinstance(reference_correction, dict):
        if reference_correction.get("tau_E_s") is not None:
            config["reference_condition_tau_E_s"] = float(
                reference_correction["tau_E_s"])
        if reference_correction.get("self_heating_ratio") is not None:
            config["reference_self_heating_ratio"] = float(
                reference_correction["self_heating_ratio"])
            config["reference_self_heating_definition"] = str(
                reference_correction.get("definition", ""))
            config["reference_self_heating_provenance"] = str(
                reference_correction.get("provenance", ""))
            config["reference_self_heating_source_status"] = str(
                reference_correction.get("status", ""))
            config["reference_self_heating_used_as_model_input"] = bool(
                reference_correction.get("use_in_current_profile_prediction", False))
    if overrides:
        config.update(overrides)
    profile = bundle["profile"]
    profile_metrics, arrays = validate_profile(profile)
    target_pressure = profile_metrics["central_pressure_Pa_reconstructed"]
    grid, psi, source, gs_info = solve_fixed_boundary_gs(
        config, target_pressure, nr=nr, nz=nz, max_iters=max_gs_iters)
    eq_metrics, field_arrays = equilibrium_metrics(
        config, grid, psi, source, gs_info)
    mhd_stability_metrics, mhd_stability_gates = mercier_ballooning_proxy(
        config, grid, field_arrays, n_thresholds=64)
    conf_metrics = confinement_metrics(config, profile, arrays)
    coolant_summary, coolant_rows = coolant_screen(config, bundle["cfd_rows"])
    write_csv(output / "coolant_screen_rows.csv", coolant_rows)

    input_root_path = Path(bundle["input_root"])
    tf_ripple_result = None
    cad_centerlines_path = input_root_path / "MHD" / "cad_a2_coil_centerlines.csv"
    cad_currents_path = input_root_path / "MHD" / "cad_a2_coil_currents.csv"
    try:
        cad_coil_bundle = load_cad_a2_tf_coil_bundle(
            cad_centerlines_path, cad_currents_path)
        tf_ripple_result = tf_ripple_screen(config, cad_coil_bundle)
    except InputBlockedError as exc:
        tf_ripple_result = {"status": "INPUT_BLOCKED", "reason": str(exc)}
    write_json(output / "tf_ripple_screen_CAD_A2_MODEL_DERIVED.json",
               tf_ripple_result)

    legacy_ripple_result = None
    legacy_centerlines_path = (
        input_root_path / "MHD" / "source_kaggle_model_derived"
        / "coil_centerlines.csv")
    legacy_currents_path = (
        input_root_path / "MHD" / "source_kaggle_model_derived"
        / "coil_currents.csv")
    if legacy_centerlines_path.exists() and legacy_currents_path.exists():
        try:
            legacy_coil_bundle = load_legacy_tf_coil_bundle(
                legacy_centerlines_path, legacy_currents_path)
            legacy_ripple_result = tf_ripple_screen(config, legacy_coil_bundle)
        except InputBlockedError as exc:
            legacy_ripple_result = {"status": "INPUT_BLOCKED", "reason": str(exc)}
        write_json(output / "tf_ripple_screen_LEGACY_BRANCH_ADVISORY.json",
                   legacy_ripple_result)

    gates = dict(eq_metrics["gates"])
    gates.update(profile_metrics["gates"])
    gates.update(conf_metrics["gates"])
    gates["input_bundle_verified"] = bool(bundle["input_audit"]["passed"])
    gates["equilibrium_numerical_pass"] = bool(
        eq_metrics["gates"]["equilibrium_numerical_pass"])
    gates["ideal_MHD_proxy_pass"] = bool(
        eq_metrics["gates"]["ideal_MHD_proxy_pass"])
    gates["profile_consistency_pass"] = bool(
        profile_metrics["profile_consistency_pass"])
    gates["confinement_numerical_screen_pass"] = bool(
        gates["equilibrium_numerical_pass"]
        and gates["ideal_MHD_proxy_pass"]
        and gates["profile_consistency_pass"])
    gates["coolant_screen_pass"] = coolant_summary["gates"].get(
        "coolant_screen_pass")
    gates["chf_correlation_screen_pass"] = coolant_summary["gates"].get(
        "chf_correlation_screen_pass")
    gates["water_property_model_pass"] = coolant_summary["gates"].get(
        "water_property_model_pass")
    gates["suydam_interchange_proxy_pass"] = mhd_stability_gates[
        "suydam_interchange_proxy_pass"]
    gates["ballooning_s_alpha_proxy_pass"] = mhd_stability_gates[
        "ballooning_s_alpha_proxy_pass"]
    gates["local_ideal_MHD_interchange_ballooning_proxy_pass"] = bool(
        gates["suydam_interchange_proxy_pass"]
        and gates["ballooning_s_alpha_proxy_pass"])
    cad_ripple_gate = bool(
        isinstance(tf_ripple_result, dict)
        and tf_ripple_result.get("controlling_gate") is True
        and tf_ripple_result.get("screening_gates", {}).get(
            "tf_coil_plasma_clearance_positive") is True
        and tf_ripple_result.get("screening_gates", {}).get(
            "tf_ripple_within_screening_threshold_at_plasma_edge") is True)
    gates["tf_ripple_screen_pass"] = cad_ripple_gate
    gates["pf_shaping_coil_input"] = "BLOCKED_REQUIRED_PF_COIL_DATA"
    gates["free_boundary_3D_MHD"] = (
        "BLOCKED_REQUIRED_PF_COILS_AND_MGRID")
    gates["nonlinear_MHD_stability_spectrum"] = "BLOCKED_EXTERNAL_SOLVER"
    gates["orbit_resolved_alpha_transport"] = "BLOCKED_EXTERNAL_SOLVER"
    gates["qualified_neutronics_tallies"] = "BLOCKED_EXTERNAL_SOLVER"
    gates["qualified_CFD_CHT"] = "BLOCKED_EXTERNAL_SOLVER"
    gates["qualified_CHF"] = "BLOCKED_EXTERNAL_SOLVER"
    gates["confinement_feasibility_screen_pass"] = bool(
        gates["equilibrium_numerical_pass"]
        and gates["ideal_MHD_proxy_pass"]
        and gates["profile_consistency_pass"]
        and gates["alpha_self_heating_pass"]
        and gates["tau_E_margin_pass"]
        and gates["reference_self_heating_reconciliation_pass"] is True
        and gates["local_ideal_MHD_interchange_ballooning_proxy_pass"]
        and gates["tf_ripple_screen_pass"])
    if bundle["source_status"] != "BUNDLE_INPUTS_VERIFIED":
        overall = "INPUT_BLOCKED"
    elif not gates["confinement_feasibility_screen_pass"]:
        overall = "SCREENING_FAIL_REVISE"
    elif gates["coolant_screen_pass"] is not True:
        overall = "SCREENING_FAIL_REVISE"
    else:
        overall = "SCREENING_PASS_WITH_EXTERNAL_BLOCKERS"

    uncertainty_summary, uncertainty_x, uncertainty_y = uncertainty_sweep(
        config, eq_metrics, conf_metrics, gates, output / "uncertainty",
        cases=cases, seed=seed)
    plot_status = make_plots(
        output, grid, field_arrays, uncertainty_x, uncertainty_y)
    write_json(output / "equilibrium_metrics.json", eq_metrics)
    write_json(output / "mhd_stability_proxy.json", mhd_stability_metrics)
    write_json(output / "confinement_metrics.json", conf_metrics)
    write_json(output / "coolant_summary.json", coolant_summary)
    write_json(output / "profile_consistency.json", profile_metrics)
    write_json(output / "gate_results.json", gates)
    np.savez_compressed(
        output / "equilibrium_fields.npz",
        R=grid["R"], Z=grid["Z"], rho=grid["rho"], psi=field_arrays["psi"],
        pressure=field_arrays["pressure"], BR=field_arrays["BR"],
        BZ=field_arrays["BZ"], BPHI=field_arrays["BPHI"],
        JPHI=field_arrays["JPHI"])
    manifest = {
        "runner_version": VERSION,
        "model_version": MODEL_VERSION,
        "label": label,
        "started_utc": utc_now(),
        "elapsed_s": time.time() - started,
        "overall_status": overall,
        "input_source_status": bundle["source_status"],
        "input_root": bundle["input_root"],
        "input_files": bundle["files"],
        "input_audit": bundle["input_audit"],
        "config": config,
        "Bosch_Hale_DT": BH_DT,
        "profile_metrics": profile_metrics,
        "equilibrium_metrics": eq_metrics,
        "mhd_stability_proxy": mhd_stability_metrics,
        "tf_ripple_screen_CAD_A2_MODEL_DERIVED": tf_ripple_result,
        "tf_ripple_screen_LEGACY_BRANCH_ADVISORY_ONLY": legacy_ripple_result,
        "confinement_metrics": conf_metrics,
        "coolant_summary": coolant_summary,
        "uncertainty_summary": uncertainty_summary,
        "gates": gates,
        "plots": plot_status,
        "limitations": [
            "Fixed-boundary axisymmetric Grad-Shafranov solve; no free-boundary coil fit.",
            "B0 plasma field, tau_E, alpha deposition, and coolant properties are explicit screening assumptions.",
            "q95 and beta_N are cylindrical/normalized proxies, not stability eigenvalue results.",
            "Suydam interchange and s-alpha ballooning checks are local, large-aspect-ratio, "
            "circular-cross-section analytic proxies (Connor-Hastie-Taylor 1978); they narrow "
            "but do not remove the nonlinear_MHD_stability_spectrum external-solver blocker.",
            "CHF is now screened with the Biasi (1967) correlation (local equilibrium quality "
            "from a Watson-correlation subcooling estimate, not a full IAPWS enthalpy balance); "
            "this narrows but does not remove the qualified_CHF external-solver blocker.",
            "The TF ripple screen is run on a separate CAD-A2 centerline with a "
            "model-derived 100-turn current rerun; the legacy branch is retained "
            "only as an advisory comparison. This is not a free-boundary poloidal "
            "equilibrium because PF/shaping coil data and mgrid are absent.",
            "D-T reaction rate uses a Maxwellian Bosch-Hale fit; no kinetic or orbit-resolved transport.",
            "The recorded reference-condition self-heating correction is compared to the current profile and is never substituted for it.",
            "Coolant rho, mu, k, and cp use verified IAPWS-IF97 Region 1, IAPWS-2008, "
            "and IAPWS-2011 industrial branches; CHF remains a Biasi correlation "
            "screen and not qualified CFD/CHF validation.",
            "No DAGMC/OpenMC nuclear-data transport, CFD/CHT, or qualified CHF correlation was run.",
        ],
        "status": "SCREENING_ARTIFACT_NOT_ENGINEERING_VALIDATION",
    }
    write_json(output / "manifest.json", manifest)
    return manifest


def zip_directory(root, destination=None):
    root = Path(root).resolve()
    destination = Path(destination).resolve() if destination else root.with_suffix(".zip")
    with zipfile.ZipFile(destination, "w", compression=zipfile.ZIP_DEFLATED,
                         compresslevel=6) as handle:
        for path in sorted(root.rglob("*")):
            if path.is_file() and path.resolve() != destination:
                handle.write(path, path.relative_to(root).as_posix())
    return destination


def main(argv=None):
    parser = argparse.ArgumentParser(description=__doc__)
    parser.add_argument("--input-root", default=None,
                        help="Path to provisional_inputs, or let the script discover it.")
    parser.add_argument("--output-root", default="tomahawk_rev24_screen")
    parser.add_argument("--smoke-only", action="store_true")
    parser.add_argument("--full-only", action="store_true")
    parser.add_argument("--smoke-cases", type=int, default=256)
    parser.add_argument("--full-cases", type=int, default=65536)
    parser.add_argument("--seed", type=int, default=20260905)
    parser.add_argument("--nr", type=int, default=161)
    parser.add_argument("--nz", type=int, default=161)
    parser.add_argument("--gs-iters", type=int, default=12000)
    parser.add_argument("--b0-t", type=float, default=DEFAULTS["B0_plasma_T"])
    parser.add_argument("--tau-e-s", type=float, default=DEFAULTS["tau_E_s"])
    parser.add_argument("--alpha-deposition", type=float,
                        default=DEFAULTS["alpha_deposition_fraction"])
    parser.add_argument("--auxiliary-absorbed-mw", type=float,
                        default=DEFAULTS["auxiliary_absorbed_MW"])
    args = parser.parse_args(argv)
    if args.smoke_only and args.full_only:
        parser.error("--smoke-only and --full-only are mutually exclusive")
    root = Path(args.output_root).resolve()
    root.mkdir(parents=True, exist_ok=True)
    overrides = {
        "B0_plasma_T": args.b0_t,
        "tau_E_s": args.tau_e_s,
        "alpha_deposition_fraction": args.alpha_deposition,
        "auxiliary_absorbed_MW": args.auxiliary_absorbed_mw,
    }
    results = {}
    try:
        if not args.full_only:
            results["smoke"] = run_native_screen(
                input_root=args.input_root, output_root=root / "smoke",
                label="smoke", cases=args.smoke_cases, seed=args.seed,
                nr=args.nr, nz=args.nz, max_gs_iters=args.gs_iters,
                overrides=overrides)
        if not args.smoke_only:
            results["full"] = run_native_screen(
                input_root=args.input_root, output_root=root / "full",
                label="full", cases=args.full_cases, seed=args.seed + 1,
                nr=args.nr, nz=args.nz, max_gs_iters=args.gs_iters,
                overrides=overrides)
    except InputBlockedError as exc:
        blocked = {
            "runner_version": VERSION,
            "created_utc": utc_now(),
            "results": {},
            "status": "INPUT_BLOCKED",
            "reason": str(exc),
            "exit_code": 3,
        }
        write_json(root / "run_manifest.json", blocked)
        archive = zip_directory(root)
        blocked["bundle_zip"] = str(archive)
        write_json(root / "run_manifest.json", blocked)
        print(json.dumps(json_safe(blocked), indent=2, allow_nan=False))
        return blocked["exit_code"]
    run_pass = all(
        result["overall_status"] == "SCREENING_PASS_WITH_EXTERNAL_BLOCKERS"
        and result["gates"]["input_bundle_verified"] is True
        and result["gates"]["confinement_feasibility_screen_pass"] is True
        and result["gates"]["coolant_screen_pass"] is True
        for result in results.values())
    summary = {
        "runner_version": VERSION,
        "created_utc": utc_now(),
        "results": {
            label: {
                "overall_status": result["overall_status"],
                "confinement_feasibility_screen_pass": result["gates"][
                    "confinement_feasibility_screen_pass"],
                "coolant_screen_pass": result["gates"]["coolant_screen_pass"],
                "full_reduced_screen_pass_percent": result[
                    "uncertainty_summary"]["all_gate_pass_percent"],
                "output": str((root / label).resolve()),
            }
            for label, result in results.items()
        },
        "status": (
            "NATIVE_SCREEN_COMPLETE_EXTERNAL_VALIDATION_STILL_REQUIRED"
            if run_pass else "NATIVE_SCREEN_FAIL_REVISE"),
        "exit_code": 0 if run_pass else 2,
    }
    write_json(root / "run_manifest.json", summary)
    archive = zip_directory(root)
    summary["bundle_zip"] = str(archive)
    write_json(root / "run_manifest.json", summary)
    print(json.dumps(json_safe(summary), indent=2, allow_nan=False))
    return summary["exit_code"]


In [ ]:
import os
from pathlib import Path
import stat
import tempfile
import zipfile


EXPECTED_RUNNER_VERSION = "TOMAHAWK-REV2.5-CAD-A2-CLOSURE-1.0-KAGGLE"
EXPECTED_INPUT_MANIFEST_SHA256 = (
    "b40af6502c2886987d6e152a154875dd54146faa9c16e882dd0f3ea75bf60c48"
)
EXPECTED_COMPLETION_STATUSES = {"SCREENING_PASS_WITH_EXTERNAL_BLOCKERS", "SCREENING_FAIL_REVISE"}

if VERSION != EXPECTED_RUNNER_VERSION:
    input_blocked(
        "embedded runner version mismatch: expected %s, found %s"
        % (EXPECTED_RUNNER_VERSION, VERSION)
    )


def unique_paths(paths):
    result = []
    for value in paths:
        path = Path(value).expanduser().resolve()
        if path.exists() and path not in result:
            result.append(path)
    return result


def manifest_candidates(root):
    root = Path(root).resolve()
    candidates = []
    if (root / "dataset_manifest.json").is_file():
        candidates.append(root)
    try:
        for manifest in sorted(root.rglob("dataset_manifest.json")):
            candidates.append(manifest.parent.resolve())
    except OSError:
        pass
    return unique_paths(candidates)


def normalized_zip_name(name):
    normalized = name.replace("\\", "/")
    while normalized.startswith("./"):
        normalized = normalized[2:]
    return normalized


def archive_manifest_members(handle):
    result = []
    for info in handle.infolist():
        normalized = normalized_zip_name(info.filename)
        if (
            normalized == "dataset_manifest.json"
            or normalized == "provisional_inputs/dataset_manifest.json"
            or normalized.endswith("/provisional_inputs/dataset_manifest.json")
        ):
            result.append(info)
    return result


def safe_extract_verified_archive(archive, target):
    archive = Path(archive).resolve()
    target = Path(target).resolve()
    target.mkdir(parents=True, exist_ok=True)
    try:
        with zipfile.ZipFile(archive) as handle:
            infos = handle.infolist()
            normalized_names = [
                normalized_zip_name(info.filename) for info in infos
            ]
            if len(normalized_names) != len(set(normalized_names)):
                input_blocked("duplicate paths in data ZIP: %s" % archive.name)
            if len(infos) > 1000:
                input_blocked("too many files in data ZIP: %s" % archive.name)
            if sum(info.file_size for info in infos) > 100 * 1024 * 1024:
                input_blocked("data ZIP expands beyond 100 MiB: %s" % archive.name)
            manifest_infos = archive_manifest_members(handle)
            if len(manifest_infos) != 1:
                input_blocked(
                    "data ZIP must contain exactly one dataset manifest: %s"
                    % archive.name
                )
            manifest_hash = hashlib.sha256(handle.read(manifest_infos[0])).hexdigest()
            if manifest_hash != EXPECTED_INPUT_MANIFEST_SHA256:
                input_blocked(
                    "data ZIP manifest mismatch in %s: expected %s, found %s"
                    % (
                        archive.name,
                        EXPECTED_INPUT_MANIFEST_SHA256,
                        manifest_hash,
                    )
                )
            for info, normalized in zip(infos, normalized_names):
                mode = (info.external_attr >> 16) & 0xFFFF
                if stat.S_ISLNK(mode):
                    input_blocked("symbolic link in data ZIP: %s" % normalized)
                destination = (target / normalized).resolve()
                if destination != target and target not in destination.parents:
                    input_blocked("unsafe path in data ZIP: %s" % normalized)
            handle.extractall(target)
    except InputBlockedError:
        raise
    except (OSError, KeyError, zipfile.BadZipFile) as exc:
        input_blocked("cannot read data ZIP %s: %s" % (archive.name, exc))


KAGGLE_INPUT_ROOT = Path(
    os.environ.get("TOMAHAWK_KAGGLE_INPUT_ROOT", "/kaggle/input")
).expanduser().resolve()
KAGGLE_WORKING_ROOT = Path(
    os.environ.get("TOMAHAWK_KAGGLE_WORKING_ROOT", "/kaggle/working")
).expanduser().resolve()
KAGGLE_WORKING_ROOT.mkdir(parents=True, exist_ok=True)

explicit_input = os.environ.get("TOMAHAWK_INPUT_ROOT")
if explicit_input:
    search_roots = unique_paths([Path(explicit_input)])
else:
    search_roots = unique_paths(
        [
            KAGGLE_INPUT_ROOT,
            Path("/kaggle/input"),
            KAGGLE_WORKING_ROOT,
            Path("/kaggle/working"),
            Path("/kaggle/models"),
            Path.cwd(),
        ]
    )

print("KAGGLE_BOOTSTRAP_START")
print("Dataset search roots:", [str(path) for path in search_roots])

direct_candidates = []
archive_candidates = []
for root in search_roots:
    direct_candidates.extend(manifest_candidates(root))
    if explicit_input:
        continue
    try:
        archive_candidates.extend(sorted(root.rglob("*.zip")))
    except OSError:
        pass

rejected_archives = []
extracted_candidates = []
for archive in unique_paths(archive_candidates):
    try:
        with zipfile.ZipFile(archive) as handle:
            members = archive_manifest_members(handle)
            if len(members) != 1:
                continue
            embedded_manifest_hash = hashlib.sha256(handle.read(members[0])).hexdigest()
            if embedded_manifest_hash != EXPECTED_INPUT_MANIFEST_SHA256:
                rejected_archives.append(
                    "%s -> manifest %s" % (archive, embedded_manifest_hash)
                )
                continue
        target = (
            KAGGLE_WORKING_ROOT
            / "tomahawk_rev24_verified_data"
            / (archive.stem + "_" + sha256(archive)[:12])
        )
        safe_extract_verified_archive(archive, target)
        extracted_candidates.extend(manifest_candidates(target))
    except (InputBlockedError, OSError, zipfile.BadZipFile) as exc:
        rejected_archives.append("%s -> %s" % (archive, exc))

candidate_roots = unique_paths([*direct_candidates, *extracted_candidates])
valid_inputs = []
rejected_inputs = []
for candidate in candidate_roots:
    manifest_path = candidate / "dataset_manifest.json"
    try:
        actual_manifest_hash = sha256(manifest_path)
        if actual_manifest_hash != EXPECTED_INPUT_MANIFEST_SHA256:
            input_blocked(
                "manifest mismatch at %s: expected %s, found %s"
                % (
                    candidate,
                    EXPECTED_INPUT_MANIFEST_SHA256,
                    actual_manifest_hash,
                )
            )
        audit = audit_input_root(candidate)
        valid_inputs.append((candidate, audit))
    except (InputBlockedError, OSError) as exc:
        rejected_inputs.append("%s -> %s" % (candidate, exc))

if not valid_inputs:
    detail = "; ".join((rejected_inputs + rejected_archives)[:6])
    if not detail:
        detail = "no dataset_manifest.json or matching data ZIP was found"
    input_blocked(
        "no verified Tomahawk Rev2.5 CAD-A2 data package under %s: %s"
        % (", ".join(str(path) for path in search_roots), detail)
    )

manifest_fingerprints = {
    audit["manifest_sha256"] for _, audit in valid_inputs
}
if manifest_fingerprints != {EXPECTED_INPUT_MANIFEST_SHA256}:
    input_blocked("ambiguous verified input manifests")

INPUT_ROOT, INPUT_AUDIT = sorted(
    valid_inputs,
    key=lambda item: (
        0 if str(item[0]).startswith(str(KAGGLE_INPUT_ROOT)) else 1,
        len(str(item[0])),
        str(item[0]),
    ),
)[0]

configured_output = os.environ.get("TOMAHAWK_OUTPUT_ROOT")
if configured_output:
    OUT = Path(configured_output).expanduser().resolve()
    OUT.mkdir(parents=True, exist_ok=True)
else:
    OUT = Path(
        tempfile.mkdtemp(
            prefix="tomahawk_rev24_screen_", dir=str(KAGGLE_WORKING_ROOT)
        )
    ).resolve()

RESULTS_ZIP = Path(
    os.environ.get(
        "TOMAHAWK_RESULTS_ZIP",
        str(KAGGLE_WORKING_ROOT / "Tomahawk_Kaggle_Results.zip"),
    )
).expanduser().resolve()

SMOKE_CASES = int(os.environ.get("TOMAHAWK_SMOKE_CASES", "256"))
FULL_CASES = int(os.environ.get("TOMAHAWK_FULL_CASES", "65536"))

print("ONE_CLICK_INPUT_VERIFIED")
print("Runner:", VERSION)
print("Inputs:", INPUT_ROOT)
print("Input manifest SHA256:", INPUT_AUDIT["manifest_sha256"])
print("Verified input files:", INPUT_AUDIT["verified_file_count"])
print("Output:", OUT)


REQUIRED_OUTPUT_NAMES = (
    "manifest.json",
    "equilibrium_metrics.json",
    "confinement_metrics.json",
    "coolant_summary.json",
    "profile_consistency.json",
    "gate_results.json",
    "equilibrium_fields.npz",
    "uncertainty/reduced_uncertainty_summary.json",
    "uncertainty/reduced_uncertainty_cases.csv",
)


def assert_run(label, result):
    run_root = OUT / label
    missing = [
        name for name in REQUIRED_OUTPUT_NAMES if not (run_root / name).is_file()
    ]
    gates = result["gates"]
    checks = {
        "runner_version": result["runner_version"] == EXPECTED_RUNNER_VERSION,
        "input_verified": gates["input_bundle_verified"] is True,
        "execution_status": result["overall_status"] in EXPECTED_COMPLETION_STATUSES,
        "GS_residual": gates["GS_residual_pass"] is True,
        "profile": gates["profile_consistency_pass"] is True,
        "reference_63_percent": gates[
            "reference_self_heating_reconciliation_pass"
        ] is True,
        "water_properties": gates["water_property_model_pass"] is True,
        "coolant_execution": gates["coolant_screen_pass"] is True,
        "CAD_A2_ripple_reported": "tf_ripple_screen_pass" in gates,
        "artifacts": not missing,
    }
    failed = [name for name, passed in checks.items() if not passed]
    if failed:
        raise RuntimeError(
            "%s_EXECUTION_FAIL: failed=%s; missing_outputs=%s"
            % (label.upper(), failed, missing)
        )
    marker = "FULL_SCREEN_COMPLETE" if label == "full" else "SMOKE_COMPLETE"
    print(marker)
    print("PHYSICS_STATUS_%s:" % label.upper(), result["overall_status"])
    print("CAD_A2_RIPPLE_GATE_%s:" % label.upper(), gates["tf_ripple_screen_pass"])
    print("PF_FREE_BOUNDARY_%s:" % label.upper(), gates["free_boundary_3D_MHD"])


smoke = run_native_screen(
    input_root=INPUT_ROOT,
    output_root=OUT / "smoke",
    label="smoke",
    cases=SMOKE_CASES,
    seed=20260905,
    nr=161,
    nz=161,
    max_gs_iters=12000,
)
assert_run("smoke", smoke)

full = run_native_screen(
    input_root=INPUT_ROOT,
    output_root=OUT / "full",
    label="full",
    cases=FULL_CASES,
    seed=20260906,
    nr=161,
    nz=161,
    max_gs_iters=12000,
)
assert_run("full", full)

confinement = full["confinement_metrics"]
coolant = full["coolant_summary"]
full_incident = coolant["scenario_summary"]["full_incident_stress"]
summary = {
    "runner_version": VERSION,
    "status": full["overall_status"],
    "markers": [
        "KAGGLE_BOOTSTRAP_START",
        "ONE_CLICK_INPUT_VERIFIED",
        "SMOKE_COMPLETE",
        "FULL_SCREEN_COMPLETE",
    ],
    "input_manifest_sha256": INPUT_AUDIT["manifest_sha256"],
    "smoke_cases": SMOKE_CASES,
    "full_cases": FULL_CASES,
    "fusion_GW_module": confinement["fusion_GW_module"],
    "active_tau_E_s": confinement["active_tau_E_screening_target_s"],
    "alpha_self_heating_ratio": confinement["alpha_self_heating_ratio"],
    "reference_tau_E_s": confinement["reference_condition_tau_E_s"],
    "reference_self_heating_ratio_recorded": confinement[
        "reference_condition_self_heating_ratio"
    ],
    "reference_self_heating_ratio_computed": confinement[
        "reference_condition_self_heating_ratio_computed"
    ],
    "tau_E_required_for_alpha_balance_s": confinement[
        "tau_E_required_for_alpha_balance_s"
    ],
    "tau_E_margin_percent": 100.0 * confinement["tau_E_margin_ratio"],
    "max_coolant_pressure_drop_kPa": coolant["max_pressure_drop_Pa"] / 1000.0,
    "max_coolant_bulk_rise_K": coolant["max_bulk_rise_K"],
    "max_coolant_wall_temperature_K": coolant["max_wall_temperature_K"],
    "full_incident_secondary_fraction": full_incident[
        "secondary_heat_removal_fraction"
    ][0],
    "full_incident_secondary_MW_per_cassette": full_incident[
        "max_secondary_heat_routed_W_per_cassette"
    ] / 1.0e6,
    "full_incident_local_MW_per_cassette": full_incident[
        "max_local_heat_load_W_per_cassette"
    ] / 1.0e6,
    "reduced_envelope_all_gate_pass_percent": full["uncertainty_summary"][
        "all_gate_pass_percent"
    ],
    "cad_a2_ripple_percent": full["tf_ripple_screen_CAD_A2_MODEL_DERIVED"][
        "ripple_at_plasma_outboard_edge"
    ]["ripple_percent"],
    "cad_a2_ripple_gate": full["gates"]["tf_ripple_screen_pass"],
    "water_properties_status": coolant["water_properties_status"],
    "pf_free_boundary_status": full["gates"]["free_boundary_3D_MHD"],
    "external_blockers": [
        "PF/shaping coil data and free-boundary 3-D equilibrium/mgrid",
        "orbit-resolved alpha transport",
        "nuclear-data neutronics",
        "qualified CFD/CHT and CHF/DNBR",
        "secondary-system capacity and deposition split",
        "hardware validation",
    ],
    "qualification": "SCREENING_ARTIFACT_NOT_ENGINEERING_VALIDATION",
}
write_json(OUT / "run_manifest.json", summary)
zip_directory(OUT, RESULTS_ZIP)

print("RUN_COMPLETE")
print(json.dumps(json_safe(summary), indent=2, allow_nan=False))
print("Results ZIP:", RESULTS_ZIP)
